# 0.0 Introduzione

Lavoro su un dataset di **soddisfazione dei passeggeri di una compagnia aerea**, nel contesto di una compagnia attenta alla qualità del servizio e all'impatto ambientale.

Il task è una **classificazione binaria**: distinguere `satisfied` da `neutral or dissatisfied`, usando 22 variabili del cliente, del volo e dei servizi. La seconda etichetta comprende anche i neutrali: non la interpreto come sola insoddisfazione.

Mantengo questo percorso:

1. **EDA** del training set, feature per feature, con distribuzioni, grafici e interpretazioni;
2. **Preprocessing**: mancanti, outlier, zeri nei rating, encoding e confronto fra scaler;
3. **Feature selection** con T-test, Chi-square e Mutual Information;
4. **Due strade richieste dalla traccia**: tutte le feature e quattro feature significative al T-test. In entrambe confronto quattro algoritmi con cinque fold, ottimizzo i migliori due e valuto i risultati sul test;
5. **Approfondimenti**: mantengo la mia selezione ibrida come confronto aggiuntivo, analizzo l'importanza delle feature e confronto il modello senza rating con quello privo anche dei ritardi effettivi;
6. **Discussione**: risultati, possibili applicazioni e limiti, separando le associazioni osservate dalle ipotesi aziendali.

Questo è un dataset di survey: il modello completo aiuta a leggere la soddisfazione a posteriori. Non misura emissioni o consumi energetici. Il test pubblico era già stato osservato nelle versioni precedenti: questa revisione migliora il metodo, ma non produce una nuova validazione esterna.

# 1.0 Setup dell'Ambiente

## 1.1 Librerie Utili

**Autore: Stefano Nocco — Progetto Finale Data Science.**

Uso Python 3.11. Le versioni con cui ho rieseguito il notebook sono riportate nella cella seguente e in `requirements.txt`. Da un terminale, nella cartella del progetto:

```bash
python -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.txt
python -m ipykernel install --user --name finale2 --display-name "Finale 2 (Python 3.11)"
```

In Jupyter seleziono questo kernel e uso **Restart Kernel and Run All**. In Colab carico i due CSV e installo queste stesse versioni prima di riavviare il runtime. Per il solo notebook, il comando equivalente è:

```python
%pip install pandas==2.2.3 numpy==2.2.6 scipy==1.15.3 scikit-learn==1.7.2 matplotlib==3.10.6 seaborn==0.13.2 plotly==6.2.0 joblib==1.6.0 kaleido==1.1.0 Jinja2==3.1.6
```

Le griglie originali sono mantenute integralmente: l'esecuzione completa può richiedere parecchio tempo. Limito il parallelismo a otto processi; `N_JOBS` può essere ridotto senza cambiare gli esperimenti. Il piccolo modulo `airline_pipeline.py` viene scritto dal notebook stesso: permette di ricaricare il modello anche in un altro processo Python.

I grafici Plotly conservano i dati interattivi nel notebook e, se Kaleido e Chrome/Chromium sono disponibili, anche una copia PNG. Per rigenerare i PNG in un ambiente privo di Chrome: `plotly_get_chrome`. La copia PNG salvata resta visibile senza collegamento a Internet; non incorporo ripetutamente la libreria JavaScript.

In [ ]:
from __future__ import annotations
import os
from pathlib import Path
import json
import hashlib
import importlib.metadata as metadata
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import seaborn as sns
from typing import List, Optional, Tuple
from IPython.display import display, Markdown

pd.options.mode.copy_on_write = True
pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda value: f'{value:,.3f}')
SEED = 0
np.random.seed(SEED)
N_JOBS = min(8, os.cpu_count() or 1)
DATA_DIR = Path('.')
RESULTS_DIR = Path('risultati')
RESULTS_DIR.mkdir(exist_ok=True)
versions = {name: metadata.version(name) for name in
            ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib', 'seaborn', 'plotly', 'joblib']}
print('Versioni:', versions)
print('Processi paralleli:', N_JOBS)
try:
    go.Figure(go.Bar(y=[1])).to_image(format='png')
    pio.renderers.default = 'plotly_mimetype+png'
    print('Grafici: formato interattivo + PNG autonomo.')
except Exception as exc:
    pio.renderers.default = 'plotly_mimetype'
    print(f'PNG non disponibile ({type(exc).__name__}); mantengo il formato interattivo.')

In [ ]:
%%writefile airline_pipeline.py
"""Trasformazioni apprese soltanto sui dati passati a fit, anche dentro la CV."""
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind, chi2_contingency
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.utils.validation import check_is_fitted

RATINGS = [
    'Inflight wifi service', 'Departure/Arrival time convenient',
    'Ease of Online booking', 'Gate location', 'Food and drink',
    'Online boarding', 'Seat comfort', 'Inflight entertainment',
    'On-board service', 'Leg room service', 'Baggage handling',
    'Checkin service', 'Inflight service', 'Cleanliness',
]
DELAYS = ['Departure Delay in Minutes', 'Arrival Delay in Minutes']
NUMERIC = ['Age', 'Flight Distance'] + DELAYS + RATINGS
CATEGORICAL = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
PRE_FLIGHT = ['Age', 'Flight Distance'] + CATEGORICAL


def welch_ranking(X, y):
    """Welch sui valori osservati; gli zeri dei rating sono esclusi come N/A."""
    labels = np.asarray(y)
    rows = []
    for column in NUMERIC:
        values = pd.to_numeric(X[column], errors='raise')
        if column in RATINGS:
            values = values.replace(0, np.nan)
        group0 = values.iloc[np.flatnonzero(labels == 0)].dropna()
        group1 = values.iloc[np.flatnonzero(labels == 1)].dropna()
        if len(group0) < 2 or len(group1) < 2 or (group0.var() == 0 and group1.var() == 0):
            stat, p = 0.0, 1.0
        else:
            stat, p = ttest_ind(group0, group1, equal_var=False)
        rows.append({'feature': column, 't_stat': abs(float(stat)), 'p_value': float(p),
                     'n0': len(group0), 'n1': len(group1)})
    return (pd.DataFrame(rows).set_index('feature')
            .sort_values(['t_stat'], ascending=False, kind='stable'))


class SurveySelector(TransformerMixin, BaseEstimator):
    """T-test obbligatorio, confronto ibrido, oppure colonne fissate a priori."""
    def __init__(self, method='ttest', k=4, alpha=0.05, columns=None):
        self.method = method
        self.k = k
        self.alpha = alpha
        self.columns = columns

    def fit(self, X, y=None):
        self.n_fit_rows_ = len(X)
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        self.class_to_business_ = False
        if self.method == 'fixed':
            self.selected_features_ = list(self.columns)
        elif self.method in ('ttest', 'hybrid'):
            self.ranking_ = welch_ranking(X, y)
            significant = self.ranking_.query('p_value < @self.alpha')
            required = self.k if self.method == 'ttest' else 2
            if len(significant) < required:
                raise ValueError(f'Solo {len(significant)} variabili significative; ne servono {required}.')
            self.selected_features_ = significant.head(required).index.tolist()
            if self.method == 'hybrid':
                scores = {}
                for col in CATEGORICAL:
                    table = pd.crosstab(X[col].fillna('Mancante'), np.asarray(y))
                    dimension = min(table.shape) - 1
                    scores[col] = (np.sqrt(chi2_contingency(table)[0] / (table.values.sum() * dimension))
                                   if dimension > 0 else 0.0)
                self.categorical_ranking_ = pd.Series(scores).sort_values(ascending=False, kind='stable')
                self.selected_features_ += self.categorical_ranking_.head(2).index.tolist()
                self.class_to_business_ = 'Class' in self.selected_features_
        else:
            raise ValueError(f'Metodo non previsto: {self.method}')
        return self

    def transform(self, X):
        check_is_fitted(self, 'selected_features_')
        result = X.loc[:, self.selected_features_].copy()
        if self.class_to_business_:
            result['Class_Business'] = (result.pop('Class') == 'Business').astype(float)
        return result

    def get_feature_names_out(self, input_features=None):
        names = list(self.selected_features_)
        if self.class_to_business_:
            names.remove('Class')
            names.append('Class_Business')
        return np.asarray(names, dtype=object)


class SurveyPreprocessor(TransformerMixin, BaseEstimator):
    """Mediane, categorie e scaler conservati insieme per inferenza da righe grezze."""
    def __init__(self, add_zero_flags=True):
        self.add_zero_flags = add_zero_flags

    def _numeric(self, X):
        result = X.loc[:, self.numeric_].astype(float).copy()
        for col in self.rating_columns_:
            result[col] = result[col].replace(0, np.nan)
        return result

    def fit(self, X, y=None):
        self.n_fit_rows_ = len(X)
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        self.categorical_ = [c for c in X if c in CATEGORICAL]
        self.numeric_ = [c for c in X if c not in self.categorical_]
        self.rating_columns_ = [c for c in self.numeric_ if c in RATINGS]
        self.zero_columns_ = [c for c in self.rating_columns_ if (X[c] == 0).any()] if self.add_zero_flags else []
        self.robust_ = [c for c in self.numeric_ if c in DELAYS]
        self.standard_ = [c for c in self.numeric_ if c not in DELAYS]
        self.imputer_ = SimpleImputer(strategy='median', keep_empty_features=True)
        numeric = pd.DataFrame(self.imputer_.fit_transform(self._numeric(X)), columns=self.numeric_, index=X.index)
        self.robust_scaler_ = RobustScaler().fit(numeric[self.robust_]) if self.robust_ else None
        self.standard_scaler_ = StandardScaler().fit(numeric[self.standard_]) if self.standard_ else None
        if self.categorical_:
            self.category_imputer_ = SimpleImputer(strategy='most_frequent', keep_empty_features=True)
            categories = self.category_imputer_.fit_transform(X[self.categorical_])
            self.encoder_ = OneHotEncoder(drop='if_binary', handle_unknown='ignore', sparse_output=False)
            self.encoder_.fit(categories)
        names = self.numeric_ + [c + '_zeroflag' for c in self.zero_columns_]
        if self.categorical_:
            names += self.encoder_.get_feature_names_out(self.categorical_).tolist()
        self.output_features_ = np.asarray(names, dtype=object)
        return self

    def transform(self, X):
        check_is_fitted(self, 'output_features_')
        result = pd.DataFrame(self.imputer_.transform(self._numeric(X)), columns=self.numeric_, index=X.index)
        if self.robust_:
            result.loc[:, self.robust_] = self.robust_scaler_.transform(result[self.robust_])
        if self.standard_:
            result.loc[:, self.standard_] = self.standard_scaler_.transform(result[self.standard_])
        for col in self.zero_columns_:
            result[col + '_zeroflag'] = (X[col] == 0).astype(float)
        if self.categorical_:
            values = self.category_imputer_.transform(X[self.categorical_])
            encoded = self.encoder_.transform(values)
            result = pd.concat([result, pd.DataFrame(encoded, index=X.index,
                                columns=self.encoder_.get_feature_names_out(self.categorical_))], axis=1)
        return result.loc[:, self.output_features_]

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, 'output_features_')
        return self.output_features_.copy()


## 1.2 Loading dei dati

I CSV sono quelli della [cartella indicata nella traccia](https://drive.google.com/drive/folders/18MMXkE98uHDuWHDgXPpwD08Y7vyEnElk?usp=sharing). Li colloco accanto al notebook, senza rinominarli. La [scheda del dataset](https://www.kaggle.com/datasets/teejmahal20/airline-passenger-satisfaction) descrive le variabili; non specifica l'unità di `Flight Distance`, che quindi mantengo come unità del dataset.

In [ ]:
for filename in ('train.csv', 'test.csv'):
    if not (DATA_DIR / filename).is_file():
        raise FileNotFoundError(f'Manca {filename}: scaricarlo dalla fonte e collocarlo accanto al notebook.')
raw_data_train = pd.read_csv(DATA_DIR / 'train.csv')
print('Training set:', raw_data_train.shape)

**Split train/test.** Mantengo i CSV già separati dalla fonte: 103.904 righe per l'addestramento e 25.976 per il test. L'EDA usa solo il train. Le trasformazioni mostrate nell'EDA sono dimostrative: per i modelli riparto dai dati grezzi e ricalcolo mediane, categorie, scaler e selezione **dentro ogni fold** della cross-validation.

Fisso algoritmi, griglie, metriche e regole di selezione prima di calcolare i nuovi punteggi sul test. Uso quest'ultimo per il confronto finale richiesto dalla traccia, senza ritoccare successivamente i modelli. Poiché il test era già noto, considero i risultati una verifica sul benchmark pubblico, non una stima su dati nuovi raccolti in produzione.

In [ ]:
raw_data_train.head()

In [ ]:
raw_data_train.info()

In [ ]:
raw_data_train.describe().T

# 2.0 Exploratory Data Analysis

## 2.1 Valori duplicati

Controllo sia le righe complete sia il contenuto senza identificativi: un ID diverso potrebbe nascondere osservazioni altrimenti identiche. Nel train non trovo duplicati con nessuno dei due criteri. Verifico inoltre l'unicità dell'ID; il controllo di sovrapposizione con il test viene eseguito nella valutazione finale.

In [ ]:
print('Righe duplicate complete:', raw_data_train.duplicated().sum())
content_columns = raw_data_train.columns.difference(['id', 'Unnamed: 0'])
print('Righe duplicate senza identificativi:', raw_data_train[content_columns].duplicated().sum())
assert raw_data_train['id'].is_unique

## 2.2 Valori mancanti

Nel train i mancanti espliciti sono 310, tutti in `Arrival Delay in Minutes` (circa 0,30% delle righe). Mantengo queste osservazioni e imputo il ritardo con la mediana, meno sensibile alla coda rispetto alla media. Nei modelli la mediana è appresa soltanto sul training di ciascun fold.

Questo conteggio riguarda i valori NaN. Lo zero nei rating richiede un controllo separato: è un valore numerico presente nel CSV, ma può rappresentare una risposta non applicabile. Lo affronto nella sezione 2.7.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from typing import Optional

def report_missing_values(
    df: pd.DataFrame,
    *,
    top_n: Optional[int] = 10,
    show_heatmap: bool = True,
    figsize: tuple[int, int] = (12, 6),
) -> pd.Series:
    """
    Analyze and visualize missing values in a DataFrame.
    
    Parameters
    ----------
    df : pd.DataFrame
        The DataFrame to analyze.
    top_n : int | None, default 10
        Number of columns to show in the percentage ranking (None → all).
    show_heatmap : bool, default True
        If True, draws a heatmap of the missing cells.
    figsize : tuple[int, int], default (12, 6)
        Figure size for the heatmap.
    
    Returns
    -------
    pd.Series
        Series with the count of NaNs per column (only those with at least
        one missing value), sorted in descending order.
    """
    # 1. Conteggio dei NaN
    missing_counts = df.isna().sum()
    missing_counts = missing_counts[missing_counts > 0].sort_values(ascending=False)

    print("Missing values per column:")
    print(missing_counts, "\n")

    # 2. Percentuali di NaN
    missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
    if top_n is not None:
        display(missing_pct.head(top_n))
    else:
        display(missing_pct)

    # 3. Heatmap opzionale
    if show_heatmap:
        plt.figure(figsize=figsize)
        sns.heatmap(df.isna(), cbar=False, yticklabels=False)
        plt.title("Map of missing values")
        plt.show()

    return missing_counts

In [ ]:
missing_counts = report_missing_values(raw_data_train, top_n=None, show_heatmap=True)

In [ ]:
raw_data_train = raw_data_train.drop(columns=["Unnamed: 0"])

In [ ]:
raw_data_train.info()

## 2.3 Esploriamo le Features

Esamino frequenze e percentuali dei valori di ogni colonna per riconoscere categorie rare, codifiche e possibili anomalie. Per le variabili continue, le frequenze dei singoli valori sono un controllo descrittivo: gli istogrammi successivi ne mostrano meglio la distribuzione. L'ID identifica le osservazioni e sarà escluso dai predittori.

In [ ]:
def summarize_feature(
    df: pd.DataFrame,
    column: str,
    /,
    *,
    dropna: bool = False,
    percent_decimals: int = 3
) -> pd.DataFrame:
    """
    Return a summary table (value, count, percent) for a single categorical column.

    Parameters
    ----------
    df : pd.DataFrame
        Source DataFrame.
    column : str
        Name of the column to summarise.
    dropna : bool, default False
        If True, exclude NaN from the counts.
    percent_decimals : int, default 2
        Number of decimal places for the percentage values.

    Returns
    -------
    pd.DataFrame
        A DataFrame with columns:
        - 'value'   : distinct values of the feature (as string)
        - 'count'   : absolute frequency
        - 'percent' : relative frequency in %
        Sorted in descending order by count.
    """
    
    if column not in df.columns:
        raise KeyError(f"Column '{column}' not found in DataFrame.")

    total_rows: int = len(df)
    vc = df[column].value_counts(dropna=dropna)

    summary = (
        pd.DataFrame(
            {
                "value": vc.index.astype(str),
                "count": vc.values,
            }
        )
        .assign(percent=lambda d: (d["count"] / total_rows * 100).round(percent_decimals))
        .sort_values(by="count", ascending=False)
        .reset_index(drop=True)
    )

    return summary

In [ ]:
features = raw_data_train.columns

In [ ]:
for feature in features:
    summary = summarize_feature(raw_data_train, feature)
    print(feature)
    print(summary)
    print("-"*30)

## 2.4 Features vs Target

Confronto ogni feature con il target per individuare associazioni utili e formulare ipotesi da approfondire.

**Come leggo i grafici.** Nelle barre, l'altezza indica il numero di osservazioni e la percentuale indica la quota di ciascuna classe entro quella categoria. Negli istogrammi delle variabili continue confronto invece conteggi: per stimare i tassi di soddisfazione per fascia servirebbe un calcolo normalizzato. Tutti questi grafici descrivono il training set; non isolano l'effetto causale di un servizio. Distinguo quindi i risultati osservati dalle possibili spiegazioni aziendali.

In [ ]:
# ------------------------ Heuristics ------------------------

def is_rating_0_5(s: pd.Series) -> bool:
    x = pd.to_numeric(s.dropna(), errors="coerce")
    if x.empty:
        return False
    return set(pd.unique(x)).issubset({0, 1, 2, 3, 4, 5})

def is_categorical(s: pd.Series, max_unique: int = 20) -> bool:
    if is_rating_0_5(s):
        return True
    if (pd.api.types.is_object_dtype(s) or
        isinstance(s.dtype, pd.CategoricalDtype) or
        pd.api.types.is_bool_dtype(s)):
        return True
    return s.nunique(dropna=True) <= max_unique

def collapse_rare_categories(s: pd.Series, top_n: int = 15) -> pd.Series:
    s = s.astype("object")
    vc = s.value_counts(dropna=False)
    if len(vc) <= top_n:
        return s.fillna("NaN")
    keep = set(vc.index[:top_n])
    out = s.where(s.isin(keep), "Other")
    return out.fillna("NaN")

# ------------------------ Data prep ------------------------

def _count_and_percent(
    x: pd.Series, y: pd.Series, classes: Optional[List] = None
) -> Tuple[pd.Index, List, pd.DataFrame, pd.DataFrame]:
    """
    Restituisce:
      - index categorie X
      - lista classi Y
      - counts_df: conteggi per (x, classe)
      - perc_df: percentuali riga (somma=100 per categoria x)
    """
    ct = pd.crosstab(x, y, dropna=False)
    if classes is not None:
        for cls in classes:
            if cls not in ct.columns:
                ct[cls] = 0
        ct = ct[classes]
    counts_df = ct.copy()
    perc_df = (ct.div(ct.sum(axis=1).replace(0, np.nan), axis=0) * 100).fillna(0.0)
    return counts_df.index, counts_df.columns.tolist(), counts_df, perc_df

# ------------------------ Plotters ------------------------

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from typing import List, Optional, Tuple

# --- utilities già definite in precedenza (riutilizzale) ---
# is_rating_0_5, collapse_rare_categories, _count_and_percent

def make_categorical_stacked_figure(
    df: pd.DataFrame,
    feature: str,
    target: str = "satisfaction",
    *,
    show_text: bool = True,       # mostra SOLO la percentuale nel segmento
    bargap: float = 0.35,         # separazione tra barre (0–5 restano “staccate”)
    top_n_cat: int = 15,          # per categoriche non 0..5
    annotate_totals: bool = True  # annota il totale in cima a ciascuna barra
) -> go.Figure:
    """
    Stacked bar: altezza = CONTEGGI reali (differenze tra valori visibili).
    Sulle barre mostriamo SOLO la percentuale della classe dentro il valore.
    Nel tooltip compaiono SIA conteggio assoluto SIA percentuale.
    """
    s_feat = df[feature]
    s_tgt  = df[target]
    classes_sorted = sorted(s_tgt.dropna().unique())

    # --- 0–5: categorie fisse in ordine naturale ---
    if is_rating_0_5(s_feat):
        vals = pd.to_numeric(s_feat, errors="coerce").where(lambda x: x.isin([0,1,2,3,4,5]))
        categories, classes, counts_df, perc_df = _count_and_percent(vals, s_tgt, classes=classes_sorted)
        all_cats = pd.Index([0,1,2,3,4,5], dtype="int64")
        counts_df = counts_df.reindex(index=all_cats, fill_value=0)
        perc_df   = perc_df.reindex(index=all_cats, fill_value=0.0)
        categories = counts_df.index
        x_categoryorder = "array"
        x_categoryarray = [0,1,2,3,4,5]
    else:
        # --- altre categoriche: collassa rare, ordina per volume totale ---
        s_feat = collapse_rare_categories(s_feat, top_n=top_n_cat)
        categories, classes, counts_df, perc_df = _count_and_percent(s_feat, s_tgt, classes=classes_sorted)
        order = counts_df.sum(axis=1).sort_values(ascending=False).index.tolist()
        counts_df = counts_df.loc[order]
        perc_df   = perc_df.loc[order]
        categories = counts_df.index
        x_categoryorder = "category descending"
        x_categoryarray = None

    fig = go.Figure()
    for cls in classes:
        y_counts = counts_df[cls].values         # altezza = conteggi
        y_perc   = perc_df[cls].values           # testo = percentuale
        customdata = np.column_stack([y_counts, y_perc])

        # 👇 SOLO percentuale visibile sulla barra
        texttemplate = "%{customdata[1]:.1f}%" if show_text else None

        fig.add_trace(go.Bar(
            x=categories,
            y=y_counts,
            name=f"Classe {cls}",
            customdata=customdata,
            texttemplate=texttemplate,            # ← niente numero assoluto nel testo
            textposition="inside" if show_text else None,
            hovertemplate=(
                f"{feature}=%{{x}}<br>{target}={cls}"
                "<br>Conteggio=%{customdata[0]}"  # ← il numero assoluto resta nel tooltip
                "<br>Percentuale=%{customdata[1]:.2f}%"
                "<extra></extra>"
            )
        ))

    # Annotazione del totale per ciascuna barra (opzionale, utile per volumi)
    totals = counts_df.sum(axis=1).values
    if annotate_totals and totals.size > 0:
        y_offset = 0.02 * totals.max() if totals.max() > 0 else 1
        for x_val, tot in zip(categories, totals):
            fig.add_annotation(
                x=x_val, y=tot + y_offset, text=str(int(tot)),
                showarrow=False, yanchor="bottom", font=dict(size=11)
            )

    fig.update_layout(
        barmode="stack",
        bargap=bargap,
        template="plotly_white",
        title=f"{feature} per {target} – stacked (conteggi) con % visibili",
        xaxis_title=feature,
        yaxis_title="Conteggio",
        legend_title=target
    )
    if x_categoryarray is not None:
        fig.update_xaxes(categoryorder=x_categoryorder, categoryarray=x_categoryarray)
    else:
        fig.update_xaxes(categoryorder=x_categoryorder)

    return fig


def make_numeric_hist_overlay_figure(
    df: pd.DataFrame,
    feature: str,
    target: str = "satisfaction",
    *,
    nbins: int = 50,
    opacity: float = 0.5,
    density: bool = False
) -> go.Figure:
    s_all = pd.to_numeric(df[feature], errors="coerce").dropna()
    if s_all.empty:
        raise ValueError(f"[{feature}] nessun dato numerico valido.")
    fmin, fmax = float(s_all.min()), float(s_all.max())
    if fmin == fmax:
        fmax = fmin + 1e-9
    bin_edges = np.linspace(fmin, fmax, nbins + 1)
    bin_size  = (fmax - fmin) / nbins

    fig = go.Figure()
    for cls in sorted(df[target].dropna().unique()):
        subset = pd.to_numeric(df.loc[df[target] == cls, feature], errors="coerce").dropna()
        if subset.empty:
            continue
        fig.add_trace(go.Histogram(
            x=subset,
            name=f"Classe {cls}",
            opacity=opacity,
            xbins=dict(start=bin_edges[0], end=bin_edges[-1], size=bin_size),
            histnorm="probability density" if density else None,
            hovertemplate=(
                f"{target}={cls}<br>{feature}=%{{x}}"
                + ("<br>Densità=%{y:.4f}" if density else "<br>Conteggio=%{y}")
                + "<extra></extra>"
            ),
        ))
    fig.update_layout(
        barmode="overlay",
        template="plotly_white",
        title=f"Distribuzione di {feature} per Classe",
        xaxis_title=feature,
        yaxis_title="Densità" if density else "Conteggio",
        legend_title=target
    )
    return fig

# ------------------------ Router (uno alla volta) ------------------------

def make_feature_by_target_figure(
    df: pd.DataFrame,
    feature: str,
    target: str = "satisfaction",
    *,
    cat_max_unique: int = 20,
    top_n_cat: int = 15,
    show_text_cat: bool = True,
    bargap_cat: float = 0.35,
    annotate_totals_cat: bool = True,
    nbins_num: int = 50,
    opacity_num: float = 0.5,
    density_num: bool = False
) -> go.Figure:
    """
    Una feature per volta:
      - Se 0–5 o categorica: stacked con altezza=conteggi + testo "N (p%)".
      - Se numerica: istogrammi sovrapposti con bins condivisi.
    """
    assert target in df.columns, f"Target '{target}' non trovato."
    s = df[feature]
    if is_rating_0_5(s) or is_categorical(s, max_unique=cat_max_unique):
        return make_categorical_stacked_figure(
            df, feature, target,
            show_text=show_text_cat,
            bargap=bargap_cat,
            top_n_cat=top_n_cat,
            annotate_totals=annotate_totals_cat
        )
    else:
        return make_numeric_hist_overlay_figure(
            df, feature, target,
            nbins=nbins_num,
            opacity=opacity_num,
            density=density_num
        )

### 2.4.1 Gender

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Gender", target="satisfaction")
fig.show()

I gruppi hanno dimensioni simili: circa 51.200 uomini e 52.700 donne. La quota di soddisfatti è rispettivamente 43,9% e 42,7%, una differenza di circa 1,2 punti percentuali.

In questo confronto il genere mostra un'associazione marginale contenuta con il target. Non ne deduco che sia irrilevante in ogni combinazione di variabili: per interpretare la differenza occorrerebbe considerare anche classe e tipo di viaggio. La sola differenza fra gruppi non giustifica un intervento aziendale basato sul genere.

### 2.4.2 Customer Type

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Customer Type", target="satisfaction")
fig.show()

I clienti `Loyal Customer` sono circa 84.900, con il 47,7% di soddisfatti; i `disloyal Customer` sono circa 19.000, con il 23,7%. La differenza osservata è di circa 24 punti percentuali e il campione contiene molti più clienti classificati come loyal.

Mantengo la variabile perché il confronto suggerisce informazione utile alla classificazione. L'associazione non dimostra che la fedeltà produca soddisfazione: potrebbero contribuire frequenza dei viaggi, condizioni del servizio o altre differenze fra i gruppi. Il dataset non documenta quando viene assegnata questa etichetta; per un impiego pre-volo andrebbe verificato che sia disponibile prima della previsione.

### 2.4.3 Age

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Age", target="satisfaction", nbins_num=60)
fig.show()

Le distribuzioni dell'età delle due classi si sovrappongono ampiamente e gran parte delle osservazioni si concentra fra 20 e 60 anni. Sono visibili differenze nella composizione delle fasce, ma l'altezza degli istogrammi risente anche della diversa numerosità delle classi.

Il grafico non basta a stabilire una relazione lineare, una forma a U o una soglia operativa. Mantengo Age fra i predittori e ne confronto il contributo con gli altri metodi di analisi; una verifica dedicata richiederebbe tassi di soddisfazione per fascia d'età e un controllo delle altre caratteristiche dei passeggeri.

### 2.4.4 Type of Travel

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Type of Travel", target="satisfaction")
fig.show()

I viaggi business rappresentano circa 71.700 osservazioni e hanno il 58,3% di soddisfatti. Nei circa 32.200 viaggi personali la quota è del 10,2%: una differenza di circa 48 punti percentuali.

Il tipo di viaggio presenta quindi una forte associazione marginale con il target. Non posso attribuire il divario a servizi prioritari o aspettative differenti, perché questi meccanismi non sono misurati direttamente. Conservo la variabile e considero che la sua informazione possa sovrapporsi a quella della classe di viaggio e del tipo di cliente.

### 2.4.5 Class

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Class", target="satisfaction")
fig.show()

La quota di soddisfatti è del 69,4% in Business, del 18,6% in Eco e del 24,6% in Eco Plus. I primi due gruppi contano circa 49.700 e 46.700 passeggeri; Eco Plus ne comprende circa 7.500.

La differenza Business–Eco è ampia, mentre quella Eco Plus–Eco è più contenuta. Questo confronto rende Class una candidata utile per la classificazione, ma non misura il beneficio di cambiare cabina a un passeggero. Per interpretare il divario servirebbe confrontare gruppi omogenei per tipo di viaggio, cliente e tratta. Nell'encoding mantengo distinte tutte e tre le classi, salvo il confronto Business/resto dichiarato nell'esperimento ibrido.

### 2.4.6 Flight Distance

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Flight Distance", target="satisfaction", nbins_num=60)
fig.show()

La distribuzione di Flight Distance è asimmetrica e presenta una coda di distanze elevate. Nelle fasce di distanza più contenuta sono numerosi i passeggeri neutrali o insoddisfatti; in alcune fasce più alte diventano più numerosi i soddisfatti. Il grafico mostra questi conteggi, senza stimare direttamente un tasso di soddisfazione per intervallo.

La distanza potrebbe essere associata anche a classe e tipo di viaggio. Non concludo quindi che volare più lontano aumenti la soddisfazione o che esista una soglia ottimale. Mantengo la variabile per il confronto predittivo e uso le unità del dataset: la fonte consultata non ne specifica l'unità fisica.

### 2.4.7 Inflight wifi service

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Inflight wifi service", target="satisfaction")
fig.show()

Per i rating Wi-Fi 1–3 la quota di soddisfatti varia fra circa il 25% e il 33%; sale al 59,9% per il rating 4 e al 99,1% per il rating 5. I rating 2 e 3 sono i più frequenti, con circa 25.800 osservazioni ciascuno. L'andamento non è strettamente monotono fra 1 e 3, ma la differenza rispetto ai rating più alti è netta.

Lo zero ha un comportamento distinto: 3.103 osservazioni e il 99,7% di soddisfatti. La scheda del dataset lo definisce «Not Applicable» per il Wi-Fi; lo tratto quindi separatamente dalla scala 1–5. Il tasso elevato non spiega perché questi passeggeri siano soddisfatti.

Il rating appartiene alla survey post-volo e può contribuire a classificarne l'esito complessivo. Non sarebbe disponibile per una previsione prima del viaggio: per questo lo escludo dall'esperimento pre-volo.

### 2.4.8 Departure/Arrival time convenient

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Departure/Arrival time convenient", target="satisfaction")
fig.show()

La quota di soddisfatti passa dal 48,7% al rating 1 al 38,8% al rating 4, per poi risalire al 42,5% al rating 5. Lo zero conta circa 5.300 osservazioni, con il 47,5% di soddisfatti. In questa variabile un giudizio più alto non corrisponde quindi a un aumento regolare della soddisfazione complessiva.

Il dato invita a verificare la codifica e la composizione dei gruppi; non dimostra che la scala sia invertita né che gli orari più convenienti peggiorino l'esperienza. Non dispongo di orari o informazioni sugli aeroporti per distinguere queste spiegazioni. Anche il significato dello zero non si può ricavare dal solo tasso osservato: applico la policy dichiarata nella sezione 2.7.

### 2.4.9 Ease of Online booking

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Ease of Online booking", target="satisfaction")
fig.show()

Per Ease of Online booking i rating 1–3 hanno quote di soddisfatti comprese fra il 30,4% e il 37,6%; i rating 4 e 5 raggiungono rispettivamente il 53,1% e il 73,8%. Il contrasto principale è dunque fra i giudizi più alti e quelli intermedi o bassi, senza monotonicità stretta lungo tutta la scala.

Lo zero comprende 4.487 osservazioni e il 66,4% di soddisfatti. Non è assimilabile al giudizio peggiore. Una risposta non applicabile o un diverso canale di prenotazione sono possibili spiegazioni, ma non sono documentate da questo grafico. Tratto lo zero secondo la policy comune dei rating e ne segnalo l'incertezza.

### 2.4.10 Gate location

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Gate location", target="satisfaction")
fig.show()

L'associazione fra Gate location e soddisfazione non è monotona: la quota di soddisfatti è circa il 49,6% al rating 1, scende al 34,7% al rating 3 e risale al 57,2% al rating 5. I rating 3 e 4 sono i più frequenti. Lo zero compare in una sola osservazione, insufficiente per descrivere un gruppo affidabile.

Il minimo nei rating intermedi suggerisce una relazione più articolata di un semplice aumento lineare. Eventuali differenze fra aeroporti o terminal restano ipotesi, perché il dataset non identifica queste condizioni. Mantengo la feature nella Strada 1; il confronto tra T-test e Mutual Information aiuta a verificare se metodi diversi colgono aspetti differenti di questo andamento.

### 2.4.11 Food and drink

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Food and drink", target="satisfaction")
fig.show()

La quota di soddisfatti è del 20,0% al rating 1, circa il 39% ai rating 2–3, il 52,5% al rating 4 e il 54,9% al rating 5. Il confronto 3–4 mostra una differenza di circa 13 punti percentuali, mentre fra 4 e 5 la differenza è più contenuta, circa 2,4 punti. Lo zero comprende 107 casi.

Il rating del servizio di ristorazione è associato al target, ma queste differenze fra gruppi non sono rendimenti attesi da un miglioramento del catering. Classe e tipo di viaggio potrebbero contribuire al risultato. Conservo i casi con zero e li tratto secondo la policy dichiarata, senza considerarli automaticamente errori perché poco frequenti.

### 2.4.12 Online boarding

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Online boarding", target="satisfaction")
fig.show()

Ai rating 1–3 di Online boarding la quota di soddisfatti resta fra l'11,5% e il 13,8%; sale al 62,3% per il rating 4 e all'87,2% per il rating 5. Il rating 4 è il più frequente, con 30.762 osservazioni. Il contrasto fra 3 e 4 è particolarmente ampio.

**Il valore 0 va letto separatamente:** conta 2.428 osservazioni, di cui 1.351 soddisfatte, pari al **55,6%**. Non appartiene quindi al gruppo dei rating con soddisfazione più bassa e l'andamento complessivo 0–5 non è monotono. La sua interpretazione come risposta non applicabile resta una policy da confermare, non una conclusione dimostrata da queste percentuali.

L'associazione rende l'esperienza di imbarco digitale un tema da approfondire. Proporrei di raccogliere dati su errori dell'app, tempi e richieste di assistenza, quindi verificare eventuali interventi con un confronto controllato. Il divario fra rating non misura quanto aumenterebbe la soddisfazione dopo un intervento.

### 2.4.13 Seat comfort

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Seat comfort", target="satisfaction")
fig.show()

Per Seat comfort la quota di soddisfatti è circa il 21–22% ai rating 1–3, il 56,1% al rating 4 e il 65,0% al rating 5. I rating 4 e 5 sono anche i più numerosi. Lo zero è presente in una sola osservazione e non permette un'interpretazione autonoma.

Il contrasto 3–4 è di circa 35 punti percentuali: è un'associazione marcata, che motiva l'inclusione della variabile nel confronto delle feature. Non equivale all'effetto di sostituire un sedile; classe e altri servizi potrebbero contribuire al divario. Essendo un rating post-volo, lo uso nei modelli basati sulla survey e lo escludo dallo scenario pre-volo.

### 2.4.14 Inflight entertainment

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Inflight entertainment", target="satisfaction")
fig.show()

La quota di soddisfatti cresce dal 14,0% al rating 1 al 27,0% al rating 3, quindi raggiunge il 61,2% al rating 4 e il 64,8% al rating 5. Il maggiore contrasto fra categorie adiacenti è quello fra 3 e 4. Lo zero compare in 14 osservazioni.

Inflight entertainment mostra un'associazione marginale forte con il target. Classe e distanza del volo sono possibili fattori da considerare per interpretarla; il grafico non consente di separarne i contributi. Mantengo gli zeri con il trattamento dichiarato e verifico la rilevanza della feature nel ranking e nei modelli.

### 2.4.15 On-board service

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="On-board service", target="satisfaction")
fig.show()

Per On-board service la quota di soddisfatti passa dal 19,4% al rating 1 al 31,6% al rating 3, poi al 53,7% al rating 4 e al 64,2% al rating 5. Il rating 4 è il più frequente, con circa 30.900 osservazioni. Lo zero compare soltanto in tre casi.

I giudizi più alti sul servizio a bordo sono associati a una maggiore soddisfazione complessiva. Il confronto 3–4 evidenzia circa 22 punti percentuali di differenza, senza dimostrare una soglia operativa del servizio. La variabile entra nel confronto con gli altri rating per valutarne l'informazione anche in presenza di predittori correlati.

### 2.4.16 Leg room service

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Leg room service", target="satisfaction")
fig.show()

Per Leg room service la quota di soddisfatti è compresa fra il 20,3% e il 27,5% ai rating 1–3, sale al 58,2% al rating 4 e al 61,4% al rating 5. Lo zero comprende 472 osservazioni, con il 35,2% di soddisfatti.

La differenza principale separa i rating 1–3 dai rating 4–5. Non la interpreto come il beneficio atteso di aumentare lo spazio per le gambe: occorrerebbe distinguere questo aspetto dalla classe e dalle altre condizioni di viaggio. La codifica dello zero viene trattata separatamente secondo la policy comune.

### 2.4.17 Baggage handling

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Baggage handling", target="satisfaction")
fig.show()

La quota di soddisfatti è circa il 30% ai rating 1–2, scende al 23,6% al rating 3, poi sale al 48,1% al rating 4 e al 61,3% al rating 5. I rating 4 e 5 raccolgono la maggior parte delle osservazioni. Non ci sono valori pari a zero.

L'andamento presenta un minimo al rating 3 e non è monotono sull'intera scala. Il grafico non permette di attribuire questo minimo a tempi di attesa, comunicazioni o aspettative: non sono variabili osservate. Conservo Baggage handling come predittore e lascio ai confronti successivi la valutazione della sua informazione rispetto agli altri servizi.

### 2.4.18 Checkin service

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Checkin service", target="satisfaction")
fig.show()

I rating 1–2 di Checkin service hanno circa il 24–25% di soddisfatti; al rating 3 la quota sale al 44,8%, al rating 4 al 46,0% e al rating 5 al 61,0%. Le differenze più visibili sono quindi fra 2 e 3 e fra 4 e 5. Lo zero compare in un solo caso.

La quota al rating 3 resta inferiore al 50%: aumenta rispetto ai rating bassi, ma i soddisfatti non diventano la maggioranza. Per interpretare l'associazione considererei anche classe e modalità di imbarco; il dataset non misura direttamente corsie prioritarie o fluidità dei processi. Mantengo la feature e tratto lo zero secondo la policy dichiarata.

### 2.4.19 Inflight service

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Inflight service", target="satisfaction")
fig.show()

Per Inflight service la quota di soddisfatti è circa il 29–30% ai rating 1–2, il 23,9% al rating 3, il 47,8% al rating 4 e il 61,0% al rating 5. Il rating 4 è il più frequente, con circa 37.900 osservazioni. Lo zero compare in tre casi.

Come per Baggage handling, il minimo al rating 3 interrompe un andamento strettamente crescente. Non posso dedurre da questo minimo specifiche reazioni dei passeggeri a cortesia, annunci o tempi di risposta. Il confronto delle correlazioni e dei modelli serve a valutare la feature insieme agli altri giudizi sul servizio.

### 2.4.20 Cleanliness

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Cleanliness", target="satisfaction")
fig.show()

La quota di soddisfatti cresce dal 19,6% al rating 1 al 21,3% al rating 2, al 43,2% al rating 3, al 53,5% al rating 4 e al 60,9% al rating 5. Lo zero compare in 12 osservazioni. Il contrasto più ampio fra categorie adiacenti è quello fra 2 e 3.

La pulizia presenta un'associazione positiva con la soddisfazione. Il 43,2% al rating 3 è vicino alla quota complessiva di soddisfatti nel train: è un riferimento descrittivo, distinto dall'accuracy della baseline maggioritaria. Le differenze fra rating non misurano il beneficio causale di un intervento di pulizia e potrebbero riflettere anche classe e altri servizi.

### 2.4.21 Departure Delay in Minutes

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Departure Delay in Minutes", target="satisfaction", nbins_num=60)
fig.show()

Departure Delay in Minutes ha una forte concentrazione vicino allo zero e una lunga coda a destra. I ritardi elevati sono poco frequenti, ma possono essere plausibili: una soglia statistica di outlier non basta per considerarli errori.

Gli istogrammi a conteggio non dimostrano soglie di soddisfazione a 15, 30 o 60 minuti. Per studiarle servirebbero tassi per intervallo e un'analisi delle altre condizioni di viaggio. Mantengo i ritardi nella Strada 1 e uso RobustScaler per la loro distribuzione asimmetrica; li escludo dallo scenario pre-volo perché si tratta di ritardi effettivi, non di previsioni disponibili alla prenotazione.

### 2.4.22 Arrival Delay in Minutes

In [ ]:
fig = make_feature_by_target_figure(raw_data_train, feature="Arrival Delay in Minutes", target="satisfaction", nbins_num=60)
fig.show()

Anche Arrival Delay in Minutes è concentrata vicino allo zero e presenta una lunga coda a destra. I 310 valori mancanti del train vengono conservati e imputati con la mediana appresa sul campione di addestramento pertinente.

Il grafico non permette di stabilire che il ritardo all'arrivo abbia un effetto maggiore di quello alla partenza. Le due variabili sono fortemente correlate, come verifico nella sezione 2.10, e il loro contributo va interpretato congiuntamente. Conservo anche questo ritardo nella Strada 1, senza ricavare soglie operative dall'istogramma; lo escludo dal confronto pre-volo.

**Sintesi dell'esplorazione.** Classe, tipo di viaggio e diversi rating mostrano differenze marcate fra gruppi; lo zero richiede un trattamento distinto dalla scala dei giudizi. Queste osservazioni guidano il preprocessing e le verifiche di selezione, ma non bastano a scegliere da sole il modello migliore. Il confronto predittivo sarà svolto con cross-validation.

## 2.5 Outliers

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats  # Import mancante!
import warnings

# Identifica colonne numeriche continue (non rating 0-5)
continuous_cols = ['Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']

# Metodo 1: IQR (Interquartile Range)
def detect_outliers_iqr(df, columns):
    outliers_summary = {}
    
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers_mask = (df[col] < lower_bound) | (df[col] > upper_bound)
        outliers_count = outliers_mask.sum()
        outliers_percentage = (outliers_count / len(df)) * 100
        
        outliers_summary[col] = {
            'count': outliers_count,
            'percentage': outliers_percentage,
            'lower_bound': lower_bound,
            'upper_bound': upper_bound
        }
    
    return outliers_summary

# Metodo 2: Z-Score
def detect_outliers_zscore(df, columns, threshold=3):
    outliers_summary = {}
    
    for col in columns:
        # Gestisci eventuali NaN
        col_data = df[col].dropna()
        z_scores = np.abs(stats.zscore(col_data))
        outliers_count = (z_scores > threshold).sum()
        outliers_percentage = (outliers_count / len(df)) * 100
        
        outliers_summary[col] = {
            'count': outliers_count,
            'percentage': outliers_percentage,
            'threshold': threshold
        }
    
    return outliers_summary

# Applica entrambi i metodi su raw_data_train
outliers_iqr = detect_outliers_iqr(raw_data_train, continuous_cols)
outliers_zscore = detect_outliers_zscore(raw_data_train, continuous_cols)

# Crea un DataFrame riassuntivo
outliers_results = pd.DataFrame({
    'IQR_count': [outliers_iqr[col]['count'] for col in continuous_cols],
    'IQR_%': [outliers_iqr[col]['percentage'] for col in continuous_cols],
    'Z-Score_count': [outliers_zscore[col]['count'] for col in continuous_cols],
    'Z-Score_%': [outliers_zscore[col]['percentage'] for col in continuous_cols]
}, index=continuous_cols)

print("OUTLIER ANALYSIS RESULTS:")
print("="*60)
print(outliers_results.round(2))

# Visualizzazione
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, col in enumerate(continuous_cols):
    ax = axes[idx]
    
    # Box plot con outlier evidenziati
    box_data = raw_data_train[col].dropna()
    box = ax.boxplot(box_data, orientation="vertical", patch_artist=True)
    box['boxes'][0].set_facecolor('lightblue')
    box['medians'][0].set_color('red')
    
    ax.set_title(f'{col}\nOutliers IQR: {outliers_iqr[col]["percentage"]:.1f}%')
    ax.set_ylabel('Value')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Analisi aggiuntiva: mostra i valori estremi per ogni variabile
print("\n" + "="*60)
print("ANALISI DETTAGLIATA OUTLIERS:")
print("="*60)

for col in continuous_cols:
    print(f"\n{col}:")
    print("-" * 30)
    
    # Statistiche base
    col_data = raw_data_train[col].dropna()
    print(f"  Min: {col_data.min():.2f}")
    print(f"  Q1: {col_data.quantile(0.25):.2f}")
    print(f"  Mediana: {col_data.median():.2f}")
    print(f"  Q3: {col_data.quantile(0.75):.2f}")
    print(f"  Max: {col_data.max():.2f}")
    
    # Bounds IQR
    print(f"  IQR Lower Bound: {outliers_iqr[col]['lower_bound']:.2f}")
    print(f"  IQR Upper Bound: {outliers_iqr[col]['upper_bound']:.2f}")
    
    # Percentuale outliers
    print(f"  Outliers (IQR): {outliers_iqr[col]['count']} ({outliers_iqr[col]['percentage']:.2f}%)")
    print(f"  Outliers (Z-score>3): {outliers_zscore[col]['count']} ({outliers_zscore[col]['percentage']:.2f}%)")

**Decisione operativa sugli outlier.**

- **Age** e **Flight Distance** hanno pochi valori oltre le soglie IQR/Z-score; età fino a 85 anni e distanze fino a circa 5.000 unità non bastano, da sole, a dimostrare errori. L'unità della distanza non è documentata nella scheda consultata.
- **Departure Delay** e **Arrival Delay** presentano circa il 13–14% di outlier IQR e code molto lunghe. Sono situazioni possibili, che interessa mantenere nell'analisi; gli outlier statistici non coincidono automaticamente con dati errati.

Non elimino righe con questi criteri. Confronto comunque modelli lineari e ad albero. Mantengo **RobustScaler** per i ritardi e **StandardScaler** per le altre numeriche: il primo usa mediana e IQR, meno sensibili alle code. Lo scaling non cancella gli estremi né rende una variabile normale; per gli alberi non è necessario, ma rende la preparazione comune e serve alla regressione logistica.

## 2.6 Analisi delle variabili categoriche

Gender, Customer Type e Type of Travel sono binarie; Class ha tre categorie. Nell'EDA mostro LabelEncoder per le binarie e le variabili indicatrici per Class. Nei modelli uso OneHotEncoder: una colonna per ciascuna binaria e tre per Class sul train completo. In questo modo non attribuisco un ordine numerico arbitrario a Eco, Eco Plus e Business.

Il Chi-square verifica l'associazione con il target; Cramér's V ne riassume l'intensità. Un p-value piccolo, soprattutto con molti record, non implica da solo una grande capacità predittiva. Per questo considero anche i ranking successivi e la validazione dei modelli.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# Identifica variabili categoriche
categorical_cols = ['Gender', 'Customer Type', 'Type of Travel', 'Class']

print("CATEGORICAL VARIABLES ANALYSIS:")
print("="*60)

# Analisi dettagliata per ogni variabile categorica
for col in categorical_cols:
    print(f"\n{col}:")
    print("-" * 30)
    
    # Conteggi e percentuali
    value_counts = raw_data_train[col].value_counts()
    value_percentages = raw_data_train[col].value_counts(normalize=True) * 100
    
    cat_summary = pd.DataFrame({
        'Count': value_counts,
        'Percentage': value_percentages.round(2)
    })
    
    print(cat_summary)
    print(f"Unique values: {raw_data_train[col].nunique()}")
    print(f"Missing values: {raw_data_train[col].isnull().sum()}")

print("\n" + "="*60)
print("STRATEGIA DI ENCODING CONSIGLIATA:")
print("="*60)

# Determina strategia di encoding basata sul numero di categorie
encoding_strategy = {}
for col in categorical_cols:
    n_unique = raw_data_train[col].nunique()
    if n_unique == 2:
        encoding_strategy[col] = "Label Encoding (Binary)"
    elif n_unique <= 5:
        encoding_strategy[col] = "One-Hot Encoding"
    else:
        encoding_strategy[col] = "Target Encoding or Embedding"
    
    print(f"{col}: {n_unique} categorie → {encoding_strategy[col]}")

print("\n" + "="*60)
print("IMPLEMENTAZIONE ENCODING:")
print("="*60)

# Crea una copia per preservare i dati originali
raw_data_encoded = raw_data_train.copy()

# 1. LABEL ENCODING per variabili binarie
binary_cols = [col for col in categorical_cols if raw_data_train[col].nunique() == 2]
le_dict = {}  # Dizionario per salvare i LabelEncoder per uso futuro

print("\n1. LABEL ENCODING (Variabili Binarie):")
print("-" * 30)

for col in binary_cols:
    le = LabelEncoder()
    raw_data_encoded[f'{col}_encoded'] = le.fit_transform(raw_data_train[col])
    le_dict[col] = le  # Salva l'encoder
    
    # Mostra il mapping
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    print(f"\n{col}:")
    for original, encoded in mapping.items():
        count = (raw_data_train[col] == original).sum()
        pct = (count / len(raw_data_train)) * 100
        print(f"  '{original}' → {encoded} (n={count}, {pct:.1f}%)")

# 2. ONE-HOT ENCODING per variabili multi-classe
multi_class_cols = [col for col in categorical_cols if raw_data_train[col].nunique() > 2]

print("\n2. ONE-HOT ENCODING (Variabili Multi-classe):")
print("-" * 30)

for col in multi_class_cols:
    print(f"\n{col} ({raw_data_train[col].nunique()} categorie):")
    
    # Crea dummies
    dummies = pd.get_dummies(raw_data_train[col], prefix=col, drop_first=False)
    
    # Mostra le nuove colonne create
    print(f"  Nuove colonne create: {list(dummies.columns)}")
    
    # Statistiche per ogni dummy
    for dummy_col in dummies.columns:
        count = dummies[dummy_col].sum()
        pct = (count / len(raw_data_train)) * 100
        print(f"    {dummy_col}: {count} records ({pct:.1f}%)")
    
    # Aggiungi al dataframe encoded (opzionale, per dimostrazione)
    raw_data_encoded = pd.concat([raw_data_encoded, dummies], axis=1)

# 3. ANALISI DELL'IMPATTO SULL'ESPANSIONE DELLE FEATURES
print("\n" + "="*60)
print("IMPATTO DELL'ENCODING SUL DATASET:")
print("="*60)

original_cols = len(raw_data_train.columns)
after_encoding_cols = len(raw_data_encoded.columns)
new_cols = after_encoding_cols - original_cols

print(f"Colonne originali: {original_cols}")
print(f"Colonne dopo encoding: {after_encoding_cols}")
print(f"Nuove colonne create: {new_cols}")

# 4. VERIFICA CORRELAZIONE TRA CATEGORICHE E TARGET
print("\n" + "="*60)
print("CORRELAZIONE CATEGORICHE CON TARGET:")
print("="*60)

from scipy.stats import chi2_contingency

for col in categorical_cols:
    # Crea tabella di contingenza
    contingency_table = pd.crosstab(raw_data_train[col], raw_data_train['satisfaction'])
    
    # Test Chi-quadrato
    chi2, p_value, dof, expected = chi2_contingency(contingency_table)
    
    # Calcola Cramér's V per misurare la forza dell'associazione
    n = contingency_table.sum().sum()
    min_dim = min(contingency_table.shape) - 1
    cramers_v = np.sqrt(chi2 / (n * min_dim))
    
    print(f"\n{col}:")
    print(f"  Chi-square: {chi2:.2f}")
    print(f"  P-value: {p_value:.4f}")
    print(f"  Cramér's V: {cramers_v:.3f}", end=" ")
    
    # Interpretazione della forza dell'associazione
    if cramers_v < 0.1:
        strength = "(molto debole)"
    elif cramers_v < 0.3:
        strength = "(debole)"
    elif cramers_v < 0.5:
        strength = "(moderata)"
    else:
        strength = "(forte)"
    print(strength)

# 5. VISUALIZZAZIONE DISTRIBUZIONE CATEGORICHE PER TARGET
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, col in enumerate(categorical_cols):
    ax = axes[idx]
    
    # Crea DataFrame per plotting
    plot_data = raw_data_train.groupby([col, 'satisfaction']).size().unstack()
    
    # Calcola percentuali per riga
    plot_data_pct = plot_data.div(plot_data.sum(axis=1), axis=0) * 100
    
    # Crea stacked bar plot
    plot_data_pct.plot(kind='bar', stacked=True, ax=ax, 
                       color=['coral', 'lightblue'],
                       edgecolor='black', linewidth=0.5)
    
    ax.set_title(f'{col} vs Satisfaction', fontsize=12, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Percentage (%)')
    ax.legend(title='Satisfaction', loc='best')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    
    # Aggiungi percentuali sulle barre
    for container in ax.containers:
        ax.bar_label(container, fmt='%.1f%%', label_type='center')

plt.suptitle('Distribuzione Variabili Categoriche per Satisfaction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 6. RACCOMANDAZIONI FINALI
print("\n" + "="*60)
print("RACCOMANDAZIONI PER IL TRATTAMENTO DELLE CATEGORICHE:")
print("="*60)

print("\n1. ENCODING STRATEGY:")
for col in categorical_cols:
    if col in binary_cols:
        print(f"   • {col}: Usa Label Encoding (binaria)")
    else:
        print(f"   • {col}: Usa One-Hot Encoding ({raw_data_train[col].nunique()} categorie)")

print("\n2. FEATURE IMPORTANCE:")
important_cats = []
for col in categorical_cols:
    contingency_table = pd.crosstab(raw_data_train[col], raw_data_train['satisfaction'])
    chi2, p_value, _, _ = chi2_contingency(contingency_table)
    if p_value < 0.001:
        important_cats.append(col)

print(f"   Variabili categoriche significative (p<0.001): {important_cats}")

In [ ]:
raw_data_encoded

## 2.7 Gestione dello zero nei rating

Il caso del Wi-Fi è documentato nella [scheda del dataset](https://www.kaggle.com/datasets/teejmahal20/airline-passenger-satisfaction): `0 = Not Applicable`. Nel train osservo anche un tasso di soddisfazione elevato fra questi record; quel tasso non è, da solo, una prova del significato del codice.

Per gli altri rating dichiaro un'**ipotesi di trattamento comune**, da confermare con un dizionario più completo: interpreto lo zero separatamente dalla scala 1–5. Nella Strada 1:

1. creo una flag per i rating che contengono zeri nel campione di addestramento;
2. sostituisco gli zeri con mancanti e imputo la mediana dei valori osservati 1–5;
3. mantengo le flag separatamente, senza scalarle.

Qui mostro il procedimento sull'intero train per comprenderlo. Nella CV viene riappreso solo sul training del fold. Nella Strada 2 mantengo esattamente quattro colonne numeriche: applico l'imputazione ma non aggiungo flag. Questa scelta rispetta il vincolo della traccia e rinuncia esplicitamente all'informazione separata sullo zero.

In [ ]:
# ============================================================
# 2.7 - GESTIONE DEI RATING == 0 COME "NON APPLICABILE"
# ============================================================
print("GESTIONE RATING == 0")
print("=" * 60)

# Lista dei rating su scala 0-5 (tutti quelli che ho ispezionato nell'EDA)
rating_features_all = [
    'Inflight wifi service', 'Departure/Arrival time convenient',
    'Ease of Online booking', 'Gate location', 'Food and drink',
    'Online boarding', 'Seat comfort', 'Inflight entertainment',
    'On-board service', 'Leg room service', 'Baggage handling',
    'Checkin service', 'Inflight service', 'Cleanliness'
]

# Identifico solo le feature per cui lo zero è plausibilmente "non applicabile":
# tengo quelle che hanno almeno un record con valore == 0
zero_features = [c for c in rating_features_all if (raw_data_train[c] == 0).any()]

print(f"Feature con almeno un rating == 0: {len(zero_features)}")
for c in zero_features:
    n_zero = int((raw_data_train[c] == 0).sum())
    pct = n_zero / len(raw_data_train) * 100
    print(f"  {c:40s} -> {n_zero:>5d} zeri ({pct:.2f}%)")

# Creo una versione "pulita" del training set:
#   1) flag binarie *_zeroflag
#   2) 0 -> NaN -> imputazione con mediana dei rating 1-5
raw_data_train_clean = raw_data_train.copy()
zero_flag_cols = []
median_imputation_map = {}

for c in zero_features:
    flag_name = f"{c}_zeroflag"
    raw_data_train_clean[flag_name] = (raw_data_train_clean[c] == 0).astype(int)
    zero_flag_cols.append(flag_name)

    # mediana calcolata SOLO sui valori 1-5 del training set
    median_val = raw_data_train_clean.loc[raw_data_train_clean[c] > 0, c].median()
    median_imputation_map[c] = median_val

    raw_data_train_clean[c] = raw_data_train_clean[c].replace(0, np.nan)
    raw_data_train_clean[c] = raw_data_train_clean[c].fillna(median_val)

print("\nNuove flag create:")
print(zero_flag_cols)

print("\nMappa di imputazione (mediana 1-5 del train):")
for c, v in median_imputation_map.items():
    print(f"  {c:40s} -> {v}")

# Verifica: nessun rating dovrebbe valere 0 nel dataset pulito
remaining_zeros = sum((raw_data_train_clean[c] == 0).sum() for c in zero_features)
print(f"\nRating con valore 0 dopo la pulizia: {remaining_zeros}")


## 2.8 Standardizzazione delle feature

Confronto StandardScaler, MinMaxScaler e RobustScaler per vedere come cambiano le scale. StandardScaler centra rispetto alla media e divide per la deviazione standard; MinMaxScaler usa gli estremi osservati; RobustScaler usa mediana e intervallo interquartile.

Lo scaling è utile per il confronto con la regressione logistica regolarizzata; non è necessario allo stesso modo per gli alberi. Nessuno di questi metodi rende automaticamente normale una distribuzione o elimina gli outlier. La dimostrazione seguente usa il train completo; la strategia mista adottata nei modelli viene poi appresa dentro ciascun fold.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
import warnings

print("STANDARDIZZAZIONE DELLE FEATURES")
print("="*60)

# 1. IDENTIFICAZIONE DELLE FEATURES DA SCALARE
print("\n1. IDENTIFICAZIONE FEATURES:")
print("-" * 30)

# Identifica diversi tipi di features
# a) Features numeriche continue
continuous_features = ['Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']

# b) Features di rating (0-5)
rating_features = [col for col in raw_data_train.columns 
                  if any(keyword in col.lower() for keyword in 
                  ['service', 'comfort', 'boarding', 'handling', 
                   'cleanliness', 'entertainment', 'wifi', 'convenient', 
                   'booking', 'location', 'food'])]

# c) Features categoriche (da NON scalare)
categorical_features = ['Gender', 'Customer Type', 'Type of Travel', 'Class']

# d) Features da escludere
exclude_features = ['id', 'satisfaction']

# Features totali da scalare
features_to_scale = continuous_features + rating_features

print(f"Features continue: {len(continuous_features)}")
print(f"  → {continuous_features}")
print(f"\nFeatures rating (0-5): {len(rating_features)}")
print(f"  → {rating_features}")
print(f"\nFeatures categoriche (NON scalare): {len(categorical_features)}")
print(f"  → {categorical_features}")
print(f"\nTotale features da scalare: {len(features_to_scale)}")

# 2. ANALISI PRE-SCALING
print("\n" + "="*60)
print("2. ANALISI DISTRIBUZIONE PRE-SCALING:")
print("-" * 30)

# Statistiche descrittive prima dello scaling
pre_scaling_stats = raw_data_train[features_to_scale].describe()

print("\nStatistiche principali (subset di features):")
sample_features = continuous_features + rating_features[:2]  # Mostra solo alcune per leggibilità

stats_summary = pd.DataFrame({
    'Mean': raw_data_train[sample_features].mean(),
    'Std': raw_data_train[sample_features].std(),
    'Min': raw_data_train[sample_features].min(),
    'Max': raw_data_train[sample_features].max(),
    'Range': raw_data_train[sample_features].max() - raw_data_train[sample_features].min()
})

print(stats_summary.round(2))

# Identificazione problemi di scala
print("\n" + "="*60)
print("3. PROBLEMI DI SCALA IDENTIFICATI:")
print("-" * 30)

# Calcola il rapporto tra i range
ranges = raw_data_train[features_to_scale].max() - raw_data_train[features_to_scale].min()
max_range = ranges.max()
min_range = ranges.min()

print(f"Range massimo: {max_range:.2f} (feature: {ranges.idxmax()})")
print(f"Range minimo: {min_range:.2f} (feature: {ranges.idxmin()})")
print(f"Rapporto max/min range: {max_range/min_range:.1f}x")

if max_range/min_range > 10:
    print("⚠️ FORTE DIFFERENZA DI SCALA - Scaling utile per i modelli sensibili alla scala (es. Logistic Regression).")
else:
    print("✓ Differenze di scala moderate")

# 3. CONFRONTO TRA METODI DI SCALING
print("\n" + "="*60)
print("4. CONFRONTO METODI DI SCALING:")
print("-" * 30)

# Crea copie del dataset per diversi metodi
data_standard = raw_data_train.copy()
data_minmax = raw_data_train.copy()
data_robust = raw_data_train.copy()

# Gestisci valori mancanti prima dello scaling
print("\nGestione valori mancanti prima dello scaling:")
missing_in_scale = raw_data_train[features_to_scale].isnull().sum()
if missing_in_scale.sum() > 0:
    print(f"  Trovati {missing_in_scale.sum()} valori mancanti")
    print("  → Imputazione con mediana per robustezza")
    for feature in features_to_scale:
        median_val = raw_data_train[feature].median()
        data_standard[feature] = data_standard[feature].fillna(median_val)
        data_minmax[feature] = data_minmax[feature].fillna(median_val)
        data_robust[feature] = data_robust[feature].fillna(median_val)
else:
    print("  ✓ Nessun valore mancante nelle features da scalare")

# Applica diversi scaler
scaler_standard = StandardScaler()
scaler_minmax = MinMaxScaler()
scaler_robust = RobustScaler()

data_standard[features_to_scale] = scaler_standard.fit_transform(data_standard[features_to_scale])
data_minmax[features_to_scale] = scaler_minmax.fit_transform(data_minmax[features_to_scale])
data_robust[features_to_scale] = scaler_robust.fit_transform(data_robust[features_to_scale])

# Confronto risultati
print("\n" + "="*60)
print("5. RISULTATI SCALING:")
print("-" * 30)

scaling_comparison = pd.DataFrame({
    'Original_Mean': raw_data_train[features_to_scale].mean(),
    'Original_Std': raw_data_train[features_to_scale].std(),
    'Standard_Mean': data_standard[features_to_scale].mean(),
    'Standard_Std': data_standard[features_to_scale].std(),
    'MinMax_Min': data_minmax[features_to_scale].min(),
    'MinMax_Max': data_minmax[features_to_scale].max(),
    'Robust_Median': data_robust[features_to_scale].median()
})

print("\nConfronto metodi (prime 5 features):")
print(scaling_comparison.head().round(3))

# Verifica qualità dello StandardScaler
print("\n" + "="*60)
print("6. VERIFICA STANDARD SCALER:")
print("-" * 30)

mean_after = data_standard[features_to_scale].mean().mean()
std_after = data_standard[features_to_scale].std().mean()

print(f"Media globale dopo StandardScaler: {mean_after:.6f} (atteso: ~0)")
print(f"Deviazione standard media dopo StandardScaler: {std_after:.6f} (atteso: ~1)")

if abs(mean_after) < 0.001 and abs(std_after - 1) < 0.01:
    print("✓ StandardScaler applicato correttamente!")
else:
    print("⚠️ Verificare l'applicazione dello StandardScaler")

# 4. VISUALIZZAZIONE DISTRIBUZIONE PRE E POST SCALING
print("\n" + "="*60)
print("7. VISUALIZZAZIONE IMPATTO SCALING:")
print("-" * 30)

# Seleziona features rappresentative per la visualizzazione
viz_features = ['Age', 'Flight Distance', 'Inflight wifi service', 'Departure Delay in Minutes']

fig, axes = plt.subplots(len(viz_features), 3, figsize=(15, len(viz_features)*3))

for idx, feature in enumerate(viz_features):
    # Original
    ax1 = axes[idx, 0]
    raw_data_train[feature].dropna().hist(bins=30, ax=ax1, color='lightblue', edgecolor='black')
    ax1.set_title(f'Original\nMean: {raw_data_train[feature].mean():.2f}, Std: {raw_data_train[feature].std():.2f}')
    ax1.set_xlabel(feature)
    ax1.set_ylabel('Frequency')
    
    # StandardScaler
    ax2 = axes[idx, 1]
    data_standard[feature].hist(bins=30, ax=ax2, color='lightgreen', edgecolor='black')
    ax2.set_title(f'StandardScaler\nMean: {data_standard[feature].mean():.2f}, Std: {data_standard[feature].std():.2f}')
    ax2.set_xlabel(f'{feature} (scaled)')
    
    # RobustScaler
    ax3 = axes[idx, 2]
    data_robust[feature].hist(bins=30, ax=ax3, color='lightsalmon', edgecolor='black')
    ax3.set_title(f'RobustScaler\nMedian: {data_robust[feature].median():.2f}')
    ax3.set_xlabel(f'{feature} (scaled)')

plt.suptitle('Confronto Metodi di Scaling', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 5. ANALISI IMPATTO OUTLIER SULLO SCALING
print("\n" + "="*60)
print("8. IMPATTO OUTLIER SULLO SCALING:")
print("-" * 30)

# Confronta scaling su features con molti outlier (Delay features)
outlier_features = ['Departure Delay in Minutes', 'Arrival Delay in Minutes']

for feature in outlier_features:
    if feature in features_to_scale:
        original_data = raw_data_train[feature].dropna()
        
        # Calcola percentuale di outlier (IQR method)
        Q1 = original_data.quantile(0.25)
        Q3 = original_data.quantile(0.75)
        IQR = Q3 - Q1
        outliers = ((original_data < Q1 - 1.5*IQR) | (original_data > Q3 + 1.5*IQR)).sum()
        outlier_pct = (outliers / len(original_data)) * 100
        
        print(f"\n{feature}:")
        print(f"  Outlier: {outlier_pct:.1f}%")
        print(f"  StandardScaler range: [{data_standard[feature].min():.2f}, {data_standard[feature].max():.2f}]")
        print(f"  RobustScaler range: [{data_robust[feature].min():.2f}, {data_robust[feature].max():.2f}]")
        
        if outlier_pct > 5:
            print(f"  → Raccomandazione: RobustScaler (resistente a outlier)")
        else:
            print(f"  → Raccomandazione: StandardScaler (distribuzione normale)")



# Applica effettivamente lo StandardScaler al dataset
print("\n" + "="*60)
print("APPLICAZIONE FINALE:")
print("-" * 30)

# Crea il dataset finale scalato
raw_data_train_scaled = raw_data_train.copy()

# Imputa valori mancanti con mediana
for feature in features_to_scale:
    if raw_data_train_scaled[feature].isnull().sum() > 0:
        median_val = raw_data_train_scaled[feature].median()
        raw_data_train_scaled[feature] = raw_data_train_scaled[feature].fillna(median_val)

# Applica StandardScaler
final_scaler = StandardScaler()
raw_data_train_scaled[features_to_scale] = final_scaler.fit_transform(raw_data_train_scaled[features_to_scale])

print(f"✓ Dataset scalato creato: raw_data_train_scaled")
print(f"  Features scalate: {len(features_to_scale)}")
print(f"  Features non scalate: {len(raw_data_train.columns) - len(features_to_scale)}")

# Dimostrazione EDA: lo scaler isolato non viene esportato.
# L'artefatto finale conterrà la pipeline completa, imputazione compresa.


### Scaling finale: strategia mista

Mantengo **RobustScaler** per i due ritardi e **StandardScaler** per le altre 16 numeriche: Age, Flight Distance e 14 rating. Le flag restano 0/1; target e identificativi non sono scalati né usati come feature.

La cella seguente è una dimostrazione sull'intero train. Nella sezione di modellazione ricostruisco questa logica in una Pipeline: ogni fold apprende i propri parametri, poi li applica alla propria validazione. Il modello finale li apprende sul train completo e li applica al test senza rifit.

In [ ]:
# ============================================================
# SCALING FINALE - STRATEGIA MISTA (Robust + Standard)
# ============================================================
print("SCALING FINALE - STRATEGIA MISTA")
print("=" * 60)

# Feature suddivise per scaler
robust_features = ['Departure Delay in Minutes', 'Arrival Delay in Minutes']
standard_features = [
    'Age', 'Flight Distance',
    'Inflight wifi service', 'Departure/Arrival time convenient',
    'Ease of Online booking', 'Gate location', 'Food and drink',
    'Online boarding', 'Seat comfort', 'Inflight entertainment',
    'On-board service', 'Leg room service', 'Baggage handling',
    'Checkin service', 'Inflight service', 'Cleanliness'
]

# Parto da raw_data_train_clean (con i 0->NaN->mediana e le _zeroflag gia' create)
# Imputo eventuali NaN residui (Arrival Delay in Minutes) con la mediana di train
raw_data_train_scaled = raw_data_train_clean.copy()
for col in robust_features + standard_features:
    if raw_data_train_scaled[col].isnull().sum() > 0:
        med = raw_data_train_scaled[col].median()
        raw_data_train_scaled[col] = raw_data_train_scaled[col].fillna(med)

# Fitting separato dei due scaler (solo su training)
robust_scaler  = RobustScaler()
standard_scaler = StandardScaler()

raw_data_train_scaled[robust_features]   = robust_scaler.fit_transform(raw_data_train_scaled[robust_features])
raw_data_train_scaled[standard_features] = standard_scaler.fit_transform(raw_data_train_scaled[standard_features])

# Sostituisco il vecchio final_scaler con un contenitore dei due scaler
final_scaler = {
    "robust": robust_scaler,
    "standard": standard_scaler,
    "robust_features": robust_features,
    "standard_features": standard_features,
    "zero_flag_cols": zero_flag_cols,
    "median_imputation_map": median_imputation_map,
}

# Aggiorno la lista features_to_scale per coerenza con il resto del notebook
features_to_scale = robust_features + standard_features

print(f"RobustScaler applicato a:  {robust_features}")
print(f"StandardScaler applicato a {len(standard_features)} feature numeriche.")
print(f"Zero-flag aggiunte come passthrough: {len(zero_flag_cols)} colonne.")

# Verifica numerica
check = pd.DataFrame({
    "feature": features_to_scale,
    "mean_after":   raw_data_train_scaled[features_to_scale].mean().round(4).values,
    "std_after":    raw_data_train_scaled[features_to_scale].std().round(4).values,
    "median_after": raw_data_train_scaled[features_to_scale].median().round(4).values,
})
print("\nVerifica statistiche dopo scaling (RobustScaler -> mediana 0; StandardScaler -> media 0, std 1):")
print(check.to_string(index=False))


## 2.9 Bilanciamento del Dataset

In [ ]:
# Analisi del bilanciamento del target
print("ANALISI BILANCIAMENTO TARGET")
print("=" * 60)

target_counts = raw_data_train["satisfaction"].value_counts()
target_pct = raw_data_train["satisfaction"].value_counts(normalize=True) * 100

print(f"\nDistribuzione del target:")
for label, count in target_counts.items():
    print(f"  {label}: {count} ({target_pct[label]:.1f}%)")

ratio = target_pct.max() / target_pct.min()
print(f"\nRapporto maggioranza/minoranza: {ratio:.2f}")

# Visualizzazione
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar plot
colors = ['#2196F3', '#FF5722']
bars = axes[0].bar(target_counts.index, target_counts.values, color=colors, edgecolor='black', linewidth=0.5)
axes[0].margins(y=0.15)
axes[0].set_title("Distribuzione del Target", fontsize=13, fontweight='bold')
axes[0].set_ylabel("Conteggio")
for bar, count, pct in zip(bars, target_counts.values, target_pct.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300,
                 f'{count}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=10)

# Pie chart
axes[1].pie(target_counts.values, labels=target_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, explode=[0.03, 0.03],
            textprops={'fontsize': 11})
axes[1].set_title("Proporzione del Target", fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nCONCLUSIONE:")
print("Il dataset e' moderatamente sbilanciato (~43% satisfied vs ~57% neutral or dissatisfied).")
print("In questa analisi mantengo la distribuzione originale senza ricampionamento.")
print("Confronto l'accuracy con la baseline e con precision, recall e F1 per entrambe le classi.")

Nel train ci sono 45.025 passeggeri soddisfatti (43,3%) e 58.879 neutrali o insoddisfatti (56,7%), con un rapporto di circa 1,31:1. Scelgo di mantenere questa distribuzione senza ricampionamento: lo sbilanciamento è moderato e non impone da solo SMOTE o undersampling.

L'accuracy resta la metrica richiesta per confrontare i modelli, ma una classe maggioritaria consente già un risultato superiore al 50% senza usare le feature. Affianco quindi la baseline maggioritaria e precision, recall e F1 per entrambe le classi. La scelta di non ricampionare è una decisione per questo esperimento, non una regola generale basata soltanto sul rapporto fra classi.

## 2.10 Correlazione tra Features

In [ ]:
# Matrice di correlazione tra features numeriche
print("ANALISI CORRELAZIONE TRA FEATURES")
print("=" * 60)

# Seleziona solo le colonne numeriche (escluso id e target encoded)
numeric_cols = features_to_scale  # 18 features (4 continue + 14 rating)

corr_matrix = raw_data_train[numeric_cols].corr()

# Heatmap
plt.figure(figsize=(16, 13))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, linewidths=0.5,
            square=True, cbar_kws={"shrink": 0.8})
plt.title("Matrice di Correlazione - Features Numeriche", fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Identifica coppie con alta correlazione (|r| > 0.7)
print("\nCOPPIE CON ALTA CORRELAZIONE (|r| > 0.7):")
print("-" * 50)

high_corr_pairs = []
for i in range(len(numeric_cols)):
    for j in range(i+1, len(numeric_cols)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.7:
            high_corr_pairs.append((numeric_cols[i], numeric_cols[j], r))
            print(f"  {numeric_cols[i]} <-> {numeric_cols[j]}: r = {r:.4f}")

if not high_corr_pairs:
    print("  Nessuna coppia con |r| > 0.7")

# Correlazione di ciascuna feature con il target (encoded)
print("\n\nCORRELAZIONE FEATURES vs TARGET:")
print("-" * 50)

target_binary = (raw_data_train["satisfaction"] == "satisfied").astype(int)
corr_with_target = raw_data_train[numeric_cols].corrwith(target_binary).sort_values(ascending=False)

for feat, r in corr_with_target.items():
    bar = "+" * int(abs(r) * 40)
    sign = "+" if r > 0 else "-"
    print(f"  {feat:40s} {sign}{abs(r):.3f}  {bar}")

# Visualizzazione
plt.figure(figsize=(10, 8))
corr_with_target.sort_values().plot(kind='barh', color=['#FF5722' if v < 0 else '#2196F3' for v in corr_with_target.sort_values()])
plt.title("Correlazione di Pearson: Features vs Target (satisfied=1)", fontsize=13, fontweight='bold')
plt.xlabel("Coefficiente di correlazione")
plt.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

print("\nCOMMENTO SULLA MULTICOLLINEARITA':")
print("Departure Delay e Arrival Delay sono fortemente correlati (r ~ 0.97).")
print("I due ritardi tendono a presentarsi insieme; la correlazione non quantifica un effetto causale.")
print("Per i modelli lineari, potrebbe essere utile rimuoverne uno; per i modelli ad albero,")
print("la predizione può restare buona, ma l'importanza può distribuirsi fra variabili correlate.")

La correlazione più alta è quella fra Departure Delay e Arrival Delay (circa 0,97): le due misure contengono informazione in larga parte sovrapposta. La matrice evidenzia anche associazioni fra alcuni rating di servizio. Con il target, Online boarding, Inflight entertainment e Seat comfort mostrano forti correlazioni positive rispetto alle altre numeriche; i ritardi hanno correlazioni negative più contenute.

Pearson descrive la componente lineare della relazione. Un coefficiente piccolo non esclude un andamento non lineare, come suggeriscono alcuni grafici dei rating. Mantengo entrambe le variabili di ritardo nella Strada 1, segnalando che la correlazione può rendere meno stabile l'interpretazione dei coefficienti e distribuire le importanze fra predittori simili.

## 2.11 Feature Selection

In [ ]:
from scipy.stats import ttest_ind, chi2_contingency
from sklearn.feature_selection import mutual_info_classif
import warnings

print("FEATURE SELECTION")
print("=" * 60)

# Target binario
target_binary = (raw_data_train["satisfaction"] == "satisfied").astype(int)
group_0 = target_binary == 0
group_1 = target_binary == 1

# -------------------------------------------------------
# 1. T-TEST per features numeriche
# -------------------------------------------------------
print("\n1. T-TEST (Features Numeriche vs Target Binario)")
print("-" * 50)

from airline_pipeline import welch_ranking
ttest_df = welch_ranking(raw_data_train, target_binary)
ttest_df["rank"] = range(1, len(ttest_df) + 1)
print(ttest_df[["t_stat", "p_value", "rank"]].to_string())

# -------------------------------------------------------
# 2. CHI-SQUARE + CRAMER'S V per features categoriche
# -------------------------------------------------------
print("\n\n2. CHI-SQUARE + CRAMER'S V (Features Categoriche)")
print("-" * 50)

chi2_results = {}
for col in categorical_cols:
    ct = pd.crosstab(raw_data_train[col], raw_data_train["satisfaction"])
    chi2, p_val, dof, _ = chi2_contingency(ct)
    n = ct.sum().sum()
    min_dim = min(ct.shape) - 1
    cramers_v = np.sqrt(chi2 / (n * min_dim))
    chi2_results[col] = {"chi2": chi2, "p_value": p_val, "cramers_v": cramers_v}

chi2_df = pd.DataFrame(chi2_results).T.sort_values("cramers_v", ascending=False)
print(chi2_df.to_string())

# -------------------------------------------------------
# 3. MUTUAL INFORMATION
# -------------------------------------------------------
print("\n\n3. MUTUAL INFORMATION (Tutte le Features)")
print("-" * 50)

# Prepara dataset completo per MI: numeriche + encoded
mi_features = features_to_scale.copy()

# Aggiungi encoded categoriche
for col in binary_cols:
    col_enc = f"{col}_encoded"
    if col_enc in raw_data_encoded.columns:
        mi_features.append(col_enc)

# Aggiungi one-hot di Class
class_dummies = [c for c in raw_data_encoded.columns if c.startswith("Class_")]
mi_features.extend(class_dummies)

X_mi = raw_data_encoded[mi_features].copy()
X_mi = X_mi.fillna(X_mi.median())

discrete_mask = [c in rating_features_all or c.endswith('_encoded') or c.startswith('Class_') for c in mi_features]
mi_scores = mutual_info_classif(X_mi, target_binary, discrete_features=discrete_mask, random_state=SEED, n_neighbors=5)
mi_df = pd.DataFrame({"feature": mi_features, "MI_score": mi_scores}).sort_values("MI_score", ascending=False)
mi_df["rank"] = range(1, len(mi_df) + 1)
print(mi_df.to_string(index=False))

# -------------------------------------------------------
# 4. CONFRONTO RANKING E VISUALIZZAZIONE
# -------------------------------------------------------
print("\n\n4. CONFRONTO RANKING")
print("-" * 50)

# Ranking dal T-test (top 10)
print("\nTop 10 features per T-test (|t-statistic|):")
for i, (feat, row) in enumerate(ttest_df.head(10).iterrows(), 1):
    print(f"  {i}. {feat} (|t| = {row['t_stat']:.2f})")

print("\nTop 10 features per Mutual Information:")
for i, row in mi_df.head(10).iterrows():
    print(f"  {int(row['rank'])}. {row['feature']} (MI = {row['MI_score']:.4f})")

# Visualizzazione comparativa
fig, axes = plt.subplots(1, 3, figsize=(20, 8))

# T-test
ttest_top = ttest_df.head(10)
axes[0].barh(range(len(ttest_top)), ttest_top["t_stat"].values, color='#2196F3')
axes[0].set_yticks(range(len(ttest_top)))
axes[0].set_yticklabels(ttest_top.index)
axes[0].set_xlabel("|t-statistic|")
axes[0].set_title("T-Test Ranking", fontweight='bold')
axes[0].invert_yaxis()

# Chi-square (Cramer's V)
axes[1].barh(range(len(chi2_df)), chi2_df["cramers_v"].values, color='#FF9800')
axes[1].set_yticks(range(len(chi2_df)))
axes[1].set_yticklabels(chi2_df.index)
axes[1].set_xlabel("Cramer's V")
axes[1].set_title("Chi-Square Ranking (Categoriche)", fontweight='bold')
axes[1].invert_yaxis()

# MI
mi_top = mi_df.head(10)
axes[2].barh(range(len(mi_top)), mi_top["MI_score"].values, color='#4CAF50')
axes[2].set_yticks(range(len(mi_top)))
axes[2].set_yticklabels(mi_top["feature"].values)
axes[2].set_xlabel("MI Score")
axes[2].set_title("Mutual Information Ranking", fontweight='bold')
axes[2].invert_yaxis()

plt.suptitle("Confronto Feature Selection: T-Test vs Chi-Square vs Mutual Information",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# -------------------------------------------------------
# -------------------------------------------------------
# 5. Quattro numeriche significative: vincolo della Strada 2
ALPHA = 0.05
top4_features = ttest_df.loc[ttest_df['p_value'] < ALPHA].head(4).index.tolist()
assert len(top4_features) == 4
print('Strada 2, ranking descrittivo sull’intero train:', top4_features)
print('Criterio: p < 0.05, ordine decrescente di |t|. In CV la selezione viene rifatta per fold.')
print('Un p-value numericamente pari a zero indica underflow, non probabilità matematica nulla.')
print(ttest_df.loc[top4_features].to_string(float_format=lambda value: f'{value:.6g}'))

I tre metodi rispondono a domande diverse:

- **Welch T-test** confronta le medie delle numeriche fra le due classi, senza assumere varianze uguali. Escludo mancanti e zeri dei rating secondo la policy dichiarata; non li riempio prima del test statistico. Uso `p < 0,05` e ordino le candidate per `|t|`, anche quando i p-value vanno in underflow. Con tanti record, significatività non significa automaticamente grande utilità predittiva. I rating sono ordinali: confrontarne le medie è un'approssimazione coerente con la traccia, non una prova causale.
- **Chi-square e Cramér's V** mostrano l'associazione delle categoriche con il target. Class e Type of Travel emergono nel loro gruppo; non confronto numericamente Cramér's V con t-statistiche per dichiarare una variabile superiore a tutte le altre.
- **Mutual Information** cattura anche dipendenze non lineari. Dichiaro rating e colonne codificate come discreti; Age, distanza e ritardi restano continui nell'estimatore. Qui conservo lo zero come categoria per confrontare anche l'informazione della codifica originale.

La **Strada 2** usa le quattro numeriche significative indicate dal T-test. Il ranking completo sopra è descrittivo: la selezione utilizzata da ciascun modello viene rifatta nella Pipeline, dentro ogni fold. Mantengo inoltre la mia **selezione ibrida** come esperimento distinto: due numeriche dal T-test e due categoriche dal Chi-square/Cramér's V. Quando seleziono Class, mantengo il contrasto Business contro le altre classi, come nel progetto originale: questo perde la distinzione Eco/Eco Plus ed è un limite esplicito.

**Cosa coincide e cosa cambia.** Le prime quattro numeriche del T-test sono Online boarding, Inflight entertainment, Inflight wifi service e Seat comfort. Le ritrovo tutte fra le prime sette colonne della Mutual Information, ma in ordine diverso: il Wi-Fi sale al secondo posto e fra le prime posizioni entrano anche Class e Type of Travel codificate. Il Chi-square analizza le quattro categoriche, quindi non può restituire lo stesso insieme delle numeriche; Class e Type of Travel sono le più associate in quel gruppo. La conclusione comune è che diversi rating e il contesto di viaggio contengono informazione. Per rispettare la Strada 2 applico comunque il criterio T-test, senza sostituirlo con un ranking scelto dopo aver visto le prestazioni.

## 2.12 Pipeline unificata, dai dati grezzi alla previsione

Riparto dai dati originali, senza riutilizzare gli scaler o gli encoder dimostrativi dell'EDA. La Pipeline contiene selezione, imputazione, encoding, scaling e classificatore. Durante la CV, `fit` vede soltanto il training del fold. Le categorie sconosciute sono gestite da OneHotEncoder con `handle_unknown='ignore'`: producono zeri nel relativo blocco, senza inventare una categoria appresa. Con categorie binarie questo coincide con la categoria di riferimento, quindi in un servizio reale monitorerei questi casi.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from airline_pipeline import SurveySelector, SurveyPreprocessor, NUMERIC, CATEGORICAL, PRE_FLIGHT, DELAYS

FEATURE_COLUMNS = [c for c in raw_data_train if c not in ['id', 'satisfaction', 'Unnamed: 0']]
X_train = raw_data_train[FEATURE_COLUMNS].copy()
y_train = raw_data_train['satisfaction'].map({'neutral or dissatisfied': 0, 'satisfied': 1})
assert y_train.notna().all()
assert len(FEATURE_COLUMNS) == 22
assert set(FEATURE_COLUMNS) == set(NUMERIC + CATEGORICAL)
print('Input grezzo:', X_train.shape, '| target:', y_train.value_counts().to_dict())

def make_pipeline(route, classifier):
    if route == 'S1':
        selector, indicators = 'passthrough', True
    elif route == 'S2':
        selector, indicators = SurveySelector(method='ttest', k=4, alpha=0.05), False
    elif route == 'Ibrida':
        selector, indicators = SurveySelector(method='hybrid', alpha=0.05), False
    elif route == 'Senza rating':
        selector, indicators = SurveySelector(method='fixed', columns=PRE_FLIGHT + DELAYS), False
    elif route == 'Pre-volo':
        selector, indicators = SurveySelector(method='fixed', columns=PRE_FLIGHT), False
    else:
        raise ValueError(route)
    return Pipeline([('select', selector), ('prepare', SurveyPreprocessor(add_zero_flags=indicators)),
                     ('model', clone(classifier))])

# Solo anteprima: questa istanza non viene passata a CV o GridSearch.
preview = SurveyPreprocessor().fit_transform(X_train)
assert np.isfinite(preview.to_numpy()).all()
print('Esempio trasformato S1:', preview.shape)
display(preview.head())
del preview

# 3.0 Scelta della metrica di valutazione

La classe positiva è **satisfied = 1**; la classe 0 comprende **neutral or dissatisfied**. Uso l'**accuracy come metrica principale**, in accordo con la traccia, e la affianco a indicatori che descrivono errori diversi:

| Metrica | Cosa misura | Come la uso |
| --- | --- | --- |
| Accuracy | Quota di classificazioni corrette sul totale. | Confronto e selezione dei modelli, insieme alla baseline maggioritaria. |
| Precision | Quota di veri soddisfatti fra i passeggeri previsti soddisfatti. | Valuto quante previsioni positive nascondono invece un cliente neutrale o insoddisfatto. |
| Recall | Quota di soddisfatti effettivi riconosciuti dal modello. | Valuto quanti clienti soddisfatti vengono persi dalla classe positiva. |
| F1 | Media armonica di precision e recall. | Riassumo il loro equilibrio, senza assumere che i due errori abbiano uguale costo aziendale. |
| ROC-AUC | Capacità di ordinare le due classi al variare della soglia. | Confronto la discriminazione; un'AUC elevata non dimostra che le probabilità siano calibrate. |

Un **falso positivo** è un passeggero previsto soddisfatto che è invece neutrale o insoddisfatto: in un uso di assistenza potrebbe non ricevere attenzione. Un **falso negativo** è un soddisfatto previsto neutrale o insoddisfatto: potrebbe ricevere un contatto non necessario. Il dataset non quantifica questi costi, quindi non scelgo una soglia aziendale ottimale; mantengo la decisione predefinita del classificatore e riporto le metriche di entrambe le classi.

Lo sbilanciamento moderato del train non rende sufficiente la sola accuracy. Confronto i risultati con un classificatore che predice sempre la classe maggioritaria appresa sul train. Per la selezione uso gli stessi cinque fold stratificati, che mantengono approssimativamente le proporzioni delle classi. Il miglior score di GridSearch resta una misura interna alla ricerca, potenzialmente ottimista dopo molti tentativi; non è una nested CV.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             confusion_matrix, roc_curve)
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier

print("Librerie per modellazione e metriche importate con successo.")
print(f"\nModelli che verranno confrontati:")
print("  1. Logistic Regression")
print("  2. Random Forest")
print("  3. AdaBoost")
print("  4. Gradient Boosting")
print(f"\nMetriche: Accuracy (principale), Precision, Recall, F1-Score, AUC-ROC")

# 4.0 Modelli, training e ottimizzazione

Confronto quattro algoritmi per verificare se relazioni non lineari e interazioni fra variabili migliorano la classificazione rispetto a un riferimento lineare. Random Forest e AdaBoost sono anche richiesti dalla traccia.

| Modello | Funzionamento essenziale | Motivo del confronto |
| --- | --- | --- |
| [Logistic Regression](https://scikit-learn.org/1.7/modules/generated/sklearn.linear_model.LogisticRegression.html) | Stima la probabilità tramite una funzione logistica di una combinazione lineare degli input; applica regolarizzazione. | Riferimento semplice: verifica quanta informazione si recupera senza aggiungere interazioni esplicite. |
| [Random Forest](https://scikit-learn.org/1.7/modules/generated/sklearn.ensemble.RandomForestClassifier.html) | Combina alberi addestrati su campioni bootstrap, con sottoinsiemi casuali di feature alle suddivisioni. | Può rappresentare relazioni non lineari e interazioni, riducendo la variabilità rispetto a un singolo albero. |
| [AdaBoost](https://scikit-learn.org/1.7/modules/generated/sklearn.ensemble.AdaBoostClassifier.html) | Costruisce classificatori in sequenza, aumentando il peso dei casi classificati male; qui usa alberi di profondità 1 come base. | Confronta una combinazione di regole semplici che dedica progressivamente attenzione ai casi difficili. |
| [Gradient Boosting](https://scikit-learn.org/1.7/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html) | Aggiunge alberi di regressione nella direzione che riduce la funzione di perdita, qui log-loss. | Verifica una forma di boosting con alberi più articolati e interazioni fra feature. |

I collegamenti rimandano alla documentazione scikit-learn 1.7 usata come riferimento. Queste caratteristiche motivano il confronto; la scelta effettiva dipende dai punteggi di validazione.

**Procedura comune alle due strade.** Eseguo uno spot check dei quattro algoritmi su cinque fold stratificati. Scelgo i due con accuracy media più alta e applico le griglie di iperparametri. Ogni fit comprende anche selezione e preprocessing, così la validazione del fold non determina mediane, scaler o feature. Il test rimane fuori dalla selezione di questa esecuzione.

**Cosa controllo con le griglie.** `n_estimators` regola il numero di alberi o di stadi; più stadi richiedono più calcolo e non garantiscono un miglior risultato. In Random Forest confronto anche `max_depth` e `min_samples_split`, che limitano la complessità degli alberi. Nei boosting `learning_rate` pesa il contributo di ciascuno stadio e va considerato insieme a `n_estimators`; in Gradient Boosting confronto anche la profondità degli alberi. Per Logistic Regression la griglia predisposta varia `C` (valori più piccoli significano regolarizzazione più forte) e `solver`, il metodo di ottimizzazione. Vengono eseguite soltanto le griglie dei due modelli selezionati nello spot check.

## 4.1 Strada 1: tutte le feature

Uso le 22 variabili originali, escludendo indice, ID e target. Sul train completo ottengo 37 colonne: 18 numeriche, 13 flag degli zeri e 6 colonne di encoding. Nei fold il numero di flag può cambiare se un caso rarissimo non è presente nel training: è una conseguenza dell'apprendimento locale. Questa strada conserva anche informazioni deboli considerate singolarmente, che potrebbero essere utili in combinazione.

In [ ]:
from sklearn.model_selection import cross_validate, ParameterGrid
from joblib import parallel_backend
import joblib
import time

models_dict = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=SEED),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=1),
    'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=SEED),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=SEED),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
spot_results, tuned, cv_audit = {}, {}, []

def spot_check(route):
    scores_by_model = {}
    for name, model in models_dict.items():
        print(f'{route} | spot check | {name}', flush=True)
        with parallel_backend('loky', inner_max_num_threads=1):
            result = cross_validate(make_pipeline(route, model), X_train, y_train, cv=cv,
                                    scoring='accuracy', n_jobs=N_JOBS, return_estimator=True,
                                    error_score='raise')
        scores = result['test_score']
        scores_by_model[name] = scores
        print(f'  Fold: {np.round(scores, 5)} | media {scores.mean():.5f} ± {scores.std():.5f}', flush=True)
        for fold, ((train_idx, validation_idx), estimator) in enumerate(zip(cv.split(X_train, y_train), result['estimator']), 1):
            prep = estimator.named_steps['prepare']
            assert prep.n_fit_rows_ == len(train_idx) < len(X_train)
            row = {'route': route, 'model': name, 'fold': fold,
                   'training_rows': len(train_idx), 'validation_rows': len(validation_idx),
                   'preprocessing_fit_rows': prep.n_fit_rows_,
                   'features': prep.get_feature_names_out().tolist()}
            selector = estimator.named_steps['select']
            if selector != 'passthrough':
                assert selector.n_fit_rows_ == len(train_idx)
                row['selected_raw'] = selector.selected_features_
            cv_audit.append(row)
        del result
        (RESULTS_DIR / 'audit_fold.json').write_text(json.dumps(cv_audit, indent=2))
    spot_results[route] = scores_by_model
    ranking = sorted(scores_by_model, key=lambda name: (-scores_by_model[name].mean(), name))
    fig, ax = plt.subplots(figsize=(11, 5))
    box = ax.boxplot([scores_by_model[name] for name in models_dict], tick_labels=list(models_dict), patch_artist=True)
    for patch, color in zip(box['boxes'], ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']):
        patch.set_facecolor(color); patch.set_alpha(0.65)
    ax.set_title(f'{route}: spot check, accuracy in 5 fold')
    ax.set_ylabel('Accuracy'); ax.grid(alpha=0.25)
    plt.tight_layout(); plt.show()
    print('Ranking:', ranking)
    pd.DataFrame(scores_by_model).to_csv(RESULTS_DIR / f'spot_{route}.csv', index_label='fold_0based')
    return ranking

ranking_s1 = spot_check('S1')
top2_names_s1 = ranking_s1[:2]

In [ ]:
param_grids = {
    "Logistic Regression": {
        "C": [0.01, 0.1, 1, 10],
        "solver": ["lbfgs", "liblinear"]
    },
    "Random Forest": {
        "n_estimators": [100, 200, 300],
        "max_depth": [10, 20, None],
        "min_samples_split": [2, 5]
    },
    "AdaBoost": {
        "n_estimators": [50, 100, 200],
        "learning_rate": [0.01, 0.1, 0.5, 1.0]
    },
    "Gradient Boosting": {
        "n_estimators": [100, 200, 300],
        "max_depth": [3, 5, 7],
        "learning_rate": [0.01, 0.1, 0.2]
    }
}


def tune(route, names):
    tuned[route] = {}
    for name in names:
        grid = {'model__' + key: values for key, values in param_grids[name].items()}
        print(f'{route} | GridSearch | {name}: {len(ParameterGrid(grid))} combinazioni × 5 fold', flush=True)
        start = time.perf_counter()
        search = GridSearchCV(make_pipeline(route, models_dict[name]), grid, cv=cv, scoring='accuracy',
                              n_jobs=N_JOBS, refit=True, error_score='raise', verbose=1,
                              return_train_score=False)
        with parallel_backend('loky', inner_max_num_threads=1):
            search.fit(X_train, y_train)
        tuned[route][name] = {'pipeline': search.best_estimator_, 'cv_accuracy': float(search.best_score_),
                              'params': search.best_params_, 'seconds': time.perf_counter() - start}
        print(f'  CV {search.best_score_:.6f} | {search.best_params_}', flush=True)
        pd.DataFrame(search.cv_results_).to_csv(RESULTS_DIR / f'grid_{route}_{name}.csv', index=False)
        joblib.dump(tuned, RESULTS_DIR / 'checkpoint_modelli.joblib', compress=3)
    return {name: item['pipeline'] for name, item in tuned[route].items()}

best_models_s1 = tune('S1', top2_names_s1)

In [ ]:
print('Strada 1: selezione completata usando soltanto la CV sul train.')
display(pd.DataFrame({name: {'Accuracy CV': item['cv_accuracy'], 'Parametri': item['params']}
                      for name, item in tuned['S1'].items()}).T)
print('Il test di entrambe le strade è raccolto nella sezione 5, dopo aver fissato tutti i modelli.')

## 4.2 Strada 2: quattro feature significative al T-test

La Pipeline seleziona quattro variabili numeriche con `p < 0,05`, ordinate per `|t|`, usando esclusivamente le righe e le etichette del training del fold. Imputazione e scaling avvengono dopo la selezione. Non aggiungo categoriche né flag: al classificatore arrivano **esattamente quattro colonne**. Confronto gli stessi quattro algoritmi e ottimizzo i migliori due con le stesse griglie della Strada 1.

In [ ]:
ranking_s2 = spot_check('S2')
top2_names_s2 = ranking_s2[:2]

In [ ]:
best_models_s2 = tune('S2', top2_names_s2)
for name, model in best_models_s2.items():
    columns = model.named_steps['prepare'].get_feature_names_out().tolist()
    assert len(columns) == 4
    print(name, '| quattro feature finali:', columns)
    print(model.named_steps['select'].ranking_.loc[columns].to_string(float_format=lambda value: f'{value:.6g}'))

In [ ]:
display(pd.DataFrame({name: {'Accuracy CV': item['cv_accuracy'], 'Parametri': item['params']}
                      for name, item in tuned['S2'].items()}).T)
print('Selezione nel train completo conclusa; il test non è usato per scegliere parametri o colonne.')

## 4.3 Approfondimenti conservati: selezione ibrida e scenari temporali

L'ibrida conserva la mia idea originale: due numeriche dal T-test e due categoriche dal Chi-square. Ripeto spot check dei quattro algoritmi e tuning dei migliori due. Per **Senza rating** e **Pre-volo** confronto gli stessi quattro algoritmi e ottimizzo il primo classificato, come nel controllo aggiuntivo originale. Non sostituiscono le due strade della traccia.

Le scelte sono fissate sul train prima della nuova valutazione sul test. Le griglie e i cinque fold restano gli stessi: non riduco la ricerca per questi approfondimenti.

In [ ]:
ranking_hybrid = spot_check('Ibrida')
best_models_hybrid = tune('Ibrida', ranking_hybrid[:2])
for name, pipeline in best_models_hybrid.items():
    assert len(pipeline.named_steps['prepare'].get_feature_names_out()) == 4
    print('Ibrida:', name, pipeline.named_steps['prepare'].get_feature_names_out().tolist())

ranking_without_ratings = spot_check('Senza rating')
best_models_without_ratings = tune('Senza rating', ranking_without_ratings[:1])
ranking_preflight = spot_check('Pre-volo')
best_models_preflight = tune('Pre-volo', ranking_preflight[:1])
for pipeline in best_models_preflight.values():
    assert set(pipeline.named_steps['select'].selected_features_) == set(PRE_FLIGHT)
    assert set(PRE_FLIGHT).isdisjoint(DELAYS)
print('Tutti i modelli sono fissati: passo al test finale.')

# 5.0 Valutazione finale e confronto

Tutti gli esperimenti ora hanno modelli e parametri fissati. Scelgo il modello principale fra S1 e S2 dalla migliore accuracy media in GridSearch; a parità esatta uso l'ordine del nome come criterio deterministico. Gli approfondimenti restano separati. Calcolo sul test il confronto richiesto, includendo baseline, report per classe, matrici di confusione e curve ROC. Il vincitore descrittivo sul test può essere diverso dalla scelta basata sulla CV: non riottimizzo in base a quel risultato.

In [ ]:
principal_candidates = {f'{route} - {name}': item for route in ['S1', 'S2'] for name, item in tuned[route].items()}
best_model_name = sorted(principal_candidates, key=lambda label: (-principal_candidates[label]['cv_accuracy'], label))[0]
best_overall = principal_candidates[best_model_name]['pipeline']
print('Modello principale scelto da CV:', best_model_name)

raw_data_test = pd.read_csv(DATA_DIR / 'test.csv').drop(columns=['Unnamed: 0'], errors='ignore')
X_test = raw_data_test[FEATURE_COLUMNS].copy()
y_test = raw_data_test['satisfaction'].map({'neutral or dissatisfied': 0, 'satisfied': 1})
assert y_test.notna().all()
assert raw_data_test['id'].is_unique
assert set(raw_data_train['id']).isdisjoint(raw_data_test['id'])
comparison_columns = FEATURE_COLUMNS + ['satisfaction']
train_hashes = pd.util.hash_pandas_object(raw_data_train[comparison_columns], index=False)
test_hashes = pd.util.hash_pandas_object(raw_data_test[comparison_columns], index=False)
assert set(train_hashes).isdisjoint(test_hashes)
assert not raw_data_test[comparison_columns].duplicated().any()

from sklearn.dummy import DummyClassifier
baseline = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
baseline_accuracy = accuracy_score(y_test, baseline.predict(X_test))
print(f'Baseline maggioritaria: {baseline_accuracy:.4%}')

def evaluate(label, pipeline, show_report=True):
    predictions = pipeline.predict(X_test)
    probabilities = pipeline.predict_proba(X_test)[:, 1]
    metrics = {'Accuracy': accuracy_score(y_test, predictions),
               'Precision': precision_score(y_test, predictions, zero_division=0),
               'Recall': recall_score(y_test, predictions, zero_division=0),
               'F1-Score': f1_score(y_test, predictions, zero_division=0),
               'AUC-ROC': roc_auc_score(y_test, probabilities)}
    print(label, {key: round(value, 5) for key, value in metrics.items()})
    if show_report:
        print(classification_report(y_test, predictions, target_names=['neutral or dissatisfied', 'satisfied'], zero_division=0))
        fig, ax = plt.subplots(figsize=(7, 5))
        sns.heatmap(confusion_matrix(y_test, predictions), annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['neutral or dissatisfied', 'satisfied'], yticklabels=['neutral or dissatisfied', 'satisfied'])
        ax.set_title(f'Matrice di confusione — {label}')
        ax.set_xlabel('Previsto'); ax.set_ylabel('Osservato')
        plt.tight_layout(); plt.show()
    return {**metrics, 'y_pred': predictions, 'y_proba': probabilities}

test_results = {}
for route, models in tuned.items():
    for name, item in models.items():
        test_results[f'{route} - {name}'] = evaluate(f'{route} - {name}', item['pipeline'])
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
comparison_df = pd.DataFrame({label: {key: test_results[label][key] for key in metric_names}
                             for label in principal_candidates}).T.sort_values('Accuracy', ascending=False)
comparison_df['Accuracy CV (selezione)'] = [principal_candidates[label]['cv_accuracy'] for label in comparison_df.index]
display(comparison_df.style.format('{:.5f}'))
comparison_df.to_csv(RESULTS_DIR / 'confronto_strade.csv')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, metric, color in zip(axes, ['Accuracy', 'AUC-ROC'], ['#2196F3', '#FF9800']):
    values = comparison_df[metric].sort_values()
    values.plot.barh(ax=ax, color=color)
    for index, value in enumerate(values): ax.text(value + .004, index, f'{value:.4f}', va='center')
    ax.set_xlim(0, 1.08); ax.set_xlabel(metric); ax.set_title(f'{metric} sul test — S1 e S2')
axes[0].axvline(baseline_accuracy, linestyle='--', color='gray', label='Baseline maggioritaria')
axes[0].legend(loc='lower right')
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(10, 7))
for label in principal_candidates:
    result = test_results[label]
    fpr, tpr, _ = roc_curve(y_test, result['y_proba'])
    ax.plot(fpr, tpr, label=f"{label} (AUC={result['AUC-ROC']:.4f})")
ax.plot([0, 1], [0, 1], 'k--', label='Casuale')
ax.set(xlabel='False Positive Rate', ylabel='True Positive Rate', title='ROC — confronto fra le due strade')
ax.legend(loc='lower right'); ax.grid(alpha=.25)
plt.tight_layout(); plt.show()
print('Prima posizione sul test (confronto descrittivo):', comparison_df.index[0])
print('Modello conservato per analisi ed export, scelto via CV:', best_model_name)

**Lettura del confronto.** Nello spot check di entrambe le strade emergono Random Forest e Gradient Boosting. Dopo il tuning, il modello principale scelto via CV è S1 - Gradient Boosting; la tabella precedente mostra anche i risultati degli altri candidati, senza selezionarli a posteriori in base al test.

La Strada 1 ottiene circa il 96,43% di accuracy sul test con il modello scelto via CV, contro l'85,33% della Strada 2: circa 11,09 punti percentuali di differenza. Ridurre a quattro rating rende più semplice l'input ma perde informazione predittiva in questo esperimento; cambiano anche le flag disponibili. Il confronto non misura quindi il contributo isolato di ogni feature esclusa. In S2 Random Forest supera di poco Gradient Boosting sul test (circa 0,03 punti percentuali), mentre l'ordine via CV è opposto: una differenza così piccola non basta a dichiarare una superiorità stabile e non modifica la regola di selezione.

## 5.1 Importanza delle feature del modello principale

L'analisi segue il modello scelto dalla CV. Per gli alberi mostro le importanze basate sulla riduzione di impurità: riassumono l'uso delle feature nelle suddivisioni e non stimano effetti causali. Possono favorire variabili con molte suddivisioni possibili e distribuirsi fra predittori correlati. In caso di modello lineare userei il valore assoluto dei coefficienti sulle scale trasformate. Il confronto con i ranking univariati va letto tenendo presenti queste differenze.

In [ ]:
classifier = best_overall.named_steps['model']
feature_names = best_overall.named_steps['prepare'].get_feature_names_out()
if hasattr(classifier, 'feature_importances_'):
    importance = classifier.feature_importances_
    importance_label = 'Importanza per riduzione di impurità'
else:
    importance = np.abs(classifier.coef_[0])
    importance_label = 'Valore assoluto del coefficiente'
fi = pd.DataFrame({'feature': feature_names, 'importance': importance}).sort_values('importance', ascending=False)
display(fi)
fig, ax = plt.subplots(figsize=(11, max(6, len(fi) * .24)))
fi.sort_values('importance').plot.barh(x='feature', y='importance', ax=ax, legend=False, color='#4CAF50')
ax.set_title(f'Importanza delle feature — {best_model_name}')
ax.set_xlabel(importance_label)
plt.tight_layout(); plt.show()
print('Top 5 modello:', fi.head(5)['feature'].tolist())
print('Top 5 T-test:', ttest_df.head(5).index.tolist())
print('Ranking categoriche:', chi2_df.index.tolist())
fi.to_csv(RESULTS_DIR / 'feature_importance.csv', index=False)

**Lettura delle importanze.** Nel modello principale Online boarding ha l'importanza maggiore; seguono Type of Travel codificato, il rating Wi-Fi e la relativa flag dello zero. Il T-test evidenzia invece differenze di media nelle singole numeriche e non include le categoriche. Il modello sfrutta dunque anche informazioni assenti dalla selezione a quattro rating. Non interpreto una bassa importanza come prova che un servizio non conti per i passeggeri: una feature correlata potrebbe già rappresentarne parte dell'informazione.

## 5.2 Approfondimenti: selezione ibrida e disponibilità delle informazioni

La **selezione ibrida** combina due numeriche selezionate con T-test e due categoriche selezionate con Chi-square/Cramér's V, sempre dentro i fold. Class diventa il contrasto Business/resto, perdendo la distinzione Eco/Eco Plus. È un confronto aggiuntivo e non sostituisce la Strada 2 richiesta.

Distinguo inoltre due scenari, entrambi addestrati e ottimizzati nella sezione 4.3:

- **Senza rating:** Age, Flight Distance, Gender, Customer Type, Type of Travel, Class e i due ritardi effettivi. È un modello post-volo senza survey dei servizi.
- **Pre-volo:** soltanto le sei variabili precedenti ai ritardi. Escludo i ritardi effettivi, che non sono stime da programma. La disponibilità delle sei variabili al momento della previsione andrebbe comunque verificata: il dataset non contiene timestamp di raccolta.

Questi confronti rispondono a domande operative diverse. La differenza di accuracy descrive modelli con informazioni differenti, non l'effetto causale dei servizi o una prestazione già validata in produzione.

In [ ]:
extra_labels = [label for label in test_results if label not in principal_candidates]
extra_df = pd.DataFrame({label: {metric: test_results[label][metric] for metric in metric_names}
                        for label in extra_labels}).T
display(extra_df.style.format('{:.5f}'))
extra_df.to_csv(RESULTS_DIR / 'confronto_approfondimenti.csv')
scenario_labels = [best_model_name] + [label for label in extra_labels if label.startswith(('Senza rating -', 'Pre-volo -'))]
scenario_accuracy = pd.Series({label: test_results[label]['Accuracy'] for label in scenario_labels})
fig, ax = plt.subplots(figsize=(12, 5))
scenario_accuracy.sort_values().plot.barh(ax=ax, color=['#FF9800', '#2196F3', '#4CAF50'])
ax.axvline(baseline_accuracy, color='gray', linestyle='--', label='Baseline maggioritaria')
ax.set_xlim(0, 1.06); ax.set_xlabel('Accuracy sul test'); ax.set_title('Disponibilità delle informazioni')
ax.legend(loc='lower right')
plt.tight_layout(); plt.show()
for label in scenario_labels[1:]:
    delta = test_results[best_model_name]['Accuracy'] - test_results[label]['Accuracy']
    print(f'Differenza rispetto al modello principale — {label}: {delta * 100:.2f} punti percentuali')
print('Le differenze sono descrittive; modelli e informazioni cambiano insieme.')

# 6.0 Conclusione

Il risultato principale è che il modello con tutte le feature classifica le survey meglio del modello limitato a quattro rating, nel benchmark analizzato. La selezione T-test produce un input più compatto, ma non conserva tutta l'informazione utile alla previsione. Il confronto pre-volo raggiunge circa il 79,48% di accuracy, contro il 56,10% della baseline: resta informazione utile anche senza rating e ritardi, ma per una domanda diversa da quella post-volo.

Tutte le trasformazioni apprese dai dati sono incluse nella cross-validation. Mantengo distinta la selezione ibrida e riporto sia la scelta tramite CV sia il successivo confronto sul test pubblico già noto.

### Come interpreto i risultati

La tabella seguente è generata dai modelli realmente eseguiti, per evitare che il testo conservi numeri di una versione precedente. La scelta principale deriva dalla CV; il confronto sul test pubblico è riportato separatamente. In particolare, una percentuale di accuracy indica classificazioni corrette, non “varianza spiegata”.

Il modello completo e quello con quattro rating servono soprattutto a leggere le survey dopo il viaggio. Il confronto pre-volo risponde a una domanda diversa, perché elimina sia i rating sia i ritardi effettivi. Il campione è moderatamente sbilanciato: confronto sempre con la baseline e considero anche precision, recall, F1, AUC e report di entrambe le classi.

### Ipotesi operative per la compagnia

1. **Online boarding.** Il forte cambiamento nella quota di soddisfatti fra i rating 3 e 4 suggerisce di approfondire l'esperienza digitale. Proporrei di misurare errori dell'app, tempi e richieste di assistenza, quindi provare un miglioramento su un gruppo controllato. Il grafico non dimostra che aumentare il voto causi lo stesso aumento di soddisfazione.
2. **Esperienza Eco.** Nel train la quota di soddisfatti è circa 18,6% in Eco e 69,4% in Business. Vorrei confrontare gruppi omogenei per tipo di viaggio, tratta e cliente prima di attribuire il divario a un singolo servizio. Le differenze osservate possono guidare interviste e prove pilota, non stimano il ritorno economico di un intervento.
3. **Gestione dei ritardi.** Separerei durata del ritardo, comunicazione e assistenza ricevuta. Questo dataset contiene la durata, ma non misura tutti gli interventi della compagnia: l'idea di migliorare notifiche e supporto è da verificare con nuovi dati.

### Collegamento con l'impatto ambientale

Il brief invita a considerare anche la sostenibilità. Qui non ho carburante, emissioni, occupazione dei voli o alternative di trasporto: non posso calcolare un risparmio di CO₂ né concludere che cambiare frequenze migliori insieme soddisfazione e ambiente. Integrerei questi dati per confrontare scenari di servizio e impatto per passeggero, tenendo distinti risultati osservati e ipotesi. Anche l'unità di Flight Distance andrebbe confermata prima di usarla in indicatori fisici.

### Limiti e passi successivi

- Il test è quello pubblico già osservato; servono nuovi dati, possibilmente separati nel tempo e per contesto operativo, per stimare la generalizzazione reale.
- La selezione e il tuning usano gli stessi cinque fold: i migliori score CV sono misure interne alla ricerca, non una valutazione indipendente. Non ho eseguito una nested CV.
- Il T-test su rating ordinali è un'approssimazione richiesta dall'esercizio; la soglia 0,05 è esplorativa e non applico una correzione per test multipli. Il criterio `|t|` ordina le candidate significative ma non massimizza necessariamente la previsione multivariata.
- Per il Wi-Fi lo zero come N/A è documentato; per gli altri rating rimane una policy da confermare. L'importanza di una flag non ne dimostra il significato. I casi molto rari possono non comparire nel training di un fold.
- I due ritardi sono fortemente correlati. Mantenerli aiuta a conservare l'informazione disponibile, ma le importanze dei modelli non vanno lette come contributi causali isolati.
- L'artefatto esportato accetta dati grezzi, ma questo controllo tecnico non equivale a un sistema pronto per l'uso aziendale. Restano da verificare disponibilità dei dati, categorie nuove, stabilità nel tempo e qualità delle previsioni sui diversi gruppi.

In [ ]:
summary_rows = []
for route, models in tuned.items():
    name = sorted(models, key=lambda key: (-models[key]['cv_accuracy'], key))[0]
    label = f'{route} - {name}'
    result = test_results[label]
    summary_rows.append({'Esperimento': route, 'Modello scelto via CV': name,
                         'Accuracy CV': models[name]['cv_accuracy'], 'Accuracy test': result['Accuracy'],
                         'AUC test': result['AUC-ROC']})
summary_table = pd.DataFrame(summary_rows).set_index('Esperimento')
display(summary_table.style.format({'Accuracy CV': '{:.2%}', 'Accuracy test': '{:.2%}', 'AUC test': '{:.5f}'}))
summary_table.to_csv(RESULTS_DIR / 'sintesi_finale.csv')
s1_row, s2_row = summary_table.loc['S1'], summary_table.loc['S2']
display(Markdown(f"Con i modelli scelti tramite CV, S1 ottiene **{s1_row['Accuracy test']:.2%}** di accuracy sul test e S2 **{s2_row['Accuracy test']:.2%}**. "
                 f"La differenza è di **{(s1_row['Accuracy test'] - s2_row['Accuracy test']) * 100:.2f} punti percentuali**. "
                 f"Il confronto pre-volo ottiene **{summary_table.loc['Pre-volo', 'Accuracy test']:.2%}**, rispetto alla baseline maggioritaria **{baseline_accuracy:.2%}**. "
                 "Sono risultati del benchmark, non effetti causali né garanzie operative."))

# 7.0 Persistenza della pipeline e prova di inferenza

Salvo lo stesso modello principale scelto dalla CV, insieme a selezione, mediane, encoder e scaler. Per usarlo in un altro processo servono il modulo `airline_pipeline.py` scritto da questo notebook e le versioni riportate. Carico soltanto l'artefatto appena creato: file pickle/joblib di provenienza sconosciuta non vanno eseguiti. La prova parte da righe grezze e controlla anche mancanti e una categoria non osservata.

In [ ]:
import subprocess
import sys
import warnings

artifact_path = Path('airline_satisfaction_model.joblib')
joblib.dump(best_overall, artifact_path, compress=3)
loaded = joblib.load(artifact_path)
sample = X_test.head(5).copy()
assert np.array_equal(loaded.predict(sample), best_overall.predict(sample))
assert np.allclose(loaded.predict_proba(sample), best_overall.predict_proba(sample))
display(pd.DataFrame({'Target': y_test.head(5).values, 'Previsione': loaded.predict(sample),
                      'P(satisfied)': loaded.predict_proba(sample)[:, 1]}))

edge_sample = sample.copy()
edge_sample.loc[edge_sample.index[0], 'Arrival Delay in Minutes'] = np.nan
edge_sample.loc[edge_sample.index[1], 'Online boarding'] = 0
edge_sample.loc[edge_sample.index[2], 'Class'] = 'Categoria nuova'
with warnings.catch_warnings(record=True) as caught:
    edge_probabilities = loaded.predict_proba(edge_sample)
assert np.isfinite(edge_probabilities).all()
print('Mancanti, rating zero e categoria nuova: previsione finita; avvisi encoder:', len(caught))

sample.to_csv(RESULTS_DIR / 'campione_inferenza.csv', index=False)
fresh_code = '''import joblib, pandas as pd, numpy as np
model = joblib.load('airline_satisfaction_model.joblib')
sample = pd.read_csv('risultati/campione_inferenza.csv')
np.save('risultati/predizioni_processo_nuovo.npy', model.predict_proba(sample))
print('Ricaricamento in un processo nuovo: OK')
'''
check = subprocess.run([sys.executable, '-c', fresh_code], text=True, capture_output=True, check=True)
print(check.stdout.strip())
assert np.allclose(np.load(RESULTS_DIR / 'predizioni_processo_nuovo.npy'), loaded.predict_proba(sample))

report = {
    'selected_by_cv': best_model_name,
    'test_ranking_leader': comparison_df.index[0],
    'baseline_accuracy': baseline_accuracy,
    'versions': versions,
    'data_sha256': {name: hashlib.sha256((DATA_DIR / name).read_bytes()).hexdigest() for name in ['train.csv', 'test.csv']},
    'metrics': {label: {key: result[key] for key in metric_names} for label, result in test_results.items()},
    'cv': {route: {name: {key: value for key, value in item.items() if key != 'pipeline'}
                   for name, item in models.items()} for route, models in tuned.items()},
    's2_features': next(iter(best_models_s2.values())).named_steps['prepare'].get_feature_names_out().tolist(),
    'raw_input_features': FEATURE_COLUMNS,
    'export_feature_count': len(feature_names),
    'inference_checks': {'raw_roundtrip': True, 'missing_zero_unknown': True, 'fresh_process': True},
}
(RESULTS_DIR / 'riepilogo.json').write_text(json.dumps(report, indent=2, ensure_ascii=False))
print(f'Pipeline salvata: {artifact_path} ({artifact_path.stat().st_size / 1e6:.2f} MB)')
print('Riepilogo, griglie complete e controlli dei fold disponibili nella cartella risultati.')